In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/test_sent_emo.csv
/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/README.txt
/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/._train_splits
/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/dev_sent_emo.csv
/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/test/._output_repeated_splits_test
/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/test/output_repeated_splits_test/dia194_utt3.mp4
/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/test/output_repeated_splits_test/dia246_utt8.mp4
/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/test/output_repeated_splits_test/dia93_utt10.mp4
/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/test/output_repeated_splits_test/._dia29_utt3.mp4
/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/test/output_repeated_splits_test/dia143_utt0.mp4
/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/test/output_repeated_splits_test/dia43_utt6.mp4
/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/test/output_repeated_splits_test/dia86_utt1.mp4
/kaggle/input/meld-dat

In [6]:
# ─── 0️⃣ INSTALL & IMPORTS ────────────────────────────────────────────────
!pip install --quiet scipy scikit-learn tqdm

import os, glob, pickle
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal            import butter, filtfilt, resample_poly, find_peaks
from sklearn.ensemble        import RandomForestClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics        import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# ─── 1️⃣ BUILD & TRAIN WESAD PHYSIO MODEL ────────────────────────────────
def convert_units(sig, mod):
    if mod=='EDA':    return ((sig/65536.0)*3.0)/0.12
    if mod=='ECG':    return ((sig/65536.0)-0.5)*3000.0
    if mod=='RESP':   return ((sig/65536.0)-0.5)*100.0
    return sig

def lowpass(sig, cutoff, fs):
    nyq = fs*0.5
    if cutoff >= nyq or sig.size < 10:
        return sig
    b,a = butter(4, cutoff/nyq, btype='low')
    try:
        return filtfilt(b,a,sig)
    except:
        return sig

def downsample(sig, frm, to, mod=None):
    if mod:
        sig = convert_units(sig, mod)
    if frm == to:
        return sig
    f = lowpass(sig, to*0.5, frm)
    return resample_poly(f, to, frm)

def basic_stats(x):
    x = x.flatten()
    return [
        x.mean(), x.std(), x.min(), x.max(),
        np.percentile(x,25), np.percentile(x,50), np.percentile(x,75),
        x.ptp(), np.mean(np.abs(np.diff(x)))
    ]

def ecg_feats(x, fs):
    x = x.flatten()
    peaks,_ = find_peaks(x, distance=fs*0.5)
    rr = np.diff(peaks)/fs if len(peaks)>1 else np.array([])
    hr  = 60/rr.mean() if rr.size>0 else 0
    hrv = rr.std()    if rr.size>0 else 0
    return [hr, hrv] + basic_stats(x)

def build_physio(root="/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"):
    fs_orig, fs_ecg, fs_other = 700, 64, 4
    win, stride = fs_orig*4, fs_orig*2
    Xp, yp = [], []
    for pkl_fp in glob.glob(os.path.join(root, "S*", "S*.pkl")):
        with open(pkl_fp,'rb') as f:
            data = pickle.load(f, encoding='latin1')
        ecg, eda, resp = data['signal']['chest']['ECG'], data['signal']['chest']['EDA'], data['signal']['chest']['Resp']
        lbl700 = data['label']
        ecg_ds  = downsample(ecg,  fs_orig, fs_ecg,  'ECG')
        eda_ds  = downsample(eda,  fs_orig, fs_other,'EDA')
        resp_ds = downsample(resp, fs_orig, fs_other,'RESP')
        for i in range(0, len(lbl700)-win+1, stride):
            lab,_ = Counter(lbl700[i:i+win]).most_common(1)[0]
            if lab not in (1,2,3,4): 
                continue
            e0,e1 = int(i/fs_orig*fs_ecg), int((i+win-1)/fs_orig*fs_ecg)+1
            o0,o1 = int(i/fs_orig*fs_other), int((i+win-1)/fs_orig*fs_other)+1
            feats = (
                ecg_feats(ecg_ds[e0:e1], fs_ecg)
                + basic_stats(eda_ds[o0:o1])
                + basic_stats(resp_ds[o0:o1])
            )
            Xp.append(feats)
            yp.append(lab-1)  # zero-based
    return np.array(Xp), np.array(yp)

X_physio, y_physio = build_physio()
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    X_physio, y_physio, test_size=0.3, stratify=y_physio, random_state=42
)

rf_physio = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42
).fit(Xp_tr, yp_tr)

print("\n=== Physio-Only RF on WESAD Test ===")
print(classification_report(yp_te, rf_physio.predict(Xp_te),
      target_names=['baseline','stress','amusement','meditation']))

# save physio probs
physio_probs_tr = rf_physio.predict_proba(Xp_tr)
physio_probs_te = rf_physio.predict_proba(Xp_te)


# ─── 2️⃣ LOAD & TRAIN MELD TEXT/AUDIO MODELS ───────────────────────────────
AUDIOTEXT = "/kaggle/input/audiotext"
train_df = pd.read_csv(os.path.join(AUDIOTEXT,"train_sent_emo.csv"))
dev_df   = pd.read_csv(os.path.join(AUDIOTEXT,"dev_sent_emo.csv"))
test_df  = pd.read_csv(os.path.join(AUDIOTEXT,"test_sent_emo.csv"))

N1, N2, N3 = len(train_df), len(dev_df), len(test_df)
print(f"\n▶️ MELD splits: train={N1}, dev={N2}, test={N3}")

# assemble y arrays
if 'label' in train_df.columns:
    y_train = train_df['label'].values
    y_dev   = dev_df  ['label'].values
    y_test  = test_df ['label'].values
else:
    # fallback if using 'Emotion' strings
    emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
    y_train = train_df['Emotion'].map(emo_map).fillna(0).astype(int).values
    y_dev   = dev_df  ['Emotion'].map(emo_map).fillna(0).astype(int).values
    y_test  = test_df ['Emotion'].map(emo_map).fillna(0).astype(int).values

# load pickles
with open(os.path.join(AUDIOTEXT,"text_emotion.pkl"), 'rb')  as f: text_data  = pickle.load(f)
with open(os.path.join(AUDIOTEXT,"audio_emotion.pkl"),'rb') as f: audio_data = pickle.load(f)

def extract_meld(data, total):
    feats=[]
    for split in data:
        for _,arr in split.items():
            feats.append(arr)
    feats = np.vstack(feats)
    if feats.shape[0]>total: feats=feats[:total]
    if feats.shape[0]<total:
        pad=total-feats.shape[0]
        feats=np.vstack([feats, np.zeros((pad,feats.shape[1]))])
    return feats

# get full features
X_text_all  = extract_meld(text_data,  N1+N2+N3)
X_audio_all = extract_meld(audio_data, N1+N2+N3)

# split
X_text_tr, X_text_dev, X_text_te   = np.split(X_text_all,  [N1, N1+N2])
X_audio_tr, X_audio_dev, X_audio_te= np.split(X_audio_all, [N1, N1+N2])

# scale
sc_t = StandardScaler().fit(np.vstack([X_text_tr,X_text_dev]))
X_text_tr  = sc_t.transform(X_text_tr)
X_text_dev = sc_t.transform(X_text_dev)
X_text_te  = sc_t.transform(X_text_te)

sc_a = StandardScaler().fit(np.vstack([X_audio_tr,X_audio_dev]))
X_audio_tr  = sc_a.transform(X_audio_tr)
X_audio_dev = sc_a.transform(X_audio_dev)
X_audio_te  = sc_a.transform(X_audio_te)

# train 7-way LRs on train+dev
lr_txt = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_aud = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

lr_txt.fit(np.vstack([X_text_tr,X_text_dev]), np.hstack([y_train,y_dev]))
lr_aud.fit(np.vstack([X_audio_tr,X_audio_dev]),np.hstack([y_train,y_dev]))

print("\n=== Text-only LR on MELD Test ===")
print(classification_report(y_test, lr_txt.predict(X_text_te)))
print("\n=== Audio-only LR on MELD Test ===")
print(classification_report(y_test, lr_aud.predict(X_audio_te)))

# get dev/test probs
text_probs_dev  = lr_txt .predict_proba(X_text_dev)
audio_probs_dev = lr_aud.predict_proba(X_audio_dev)
text_probs_te   = lr_txt .predict_proba(X_text_te)
audio_probs_te  = lr_aud.predict_proba(X_audio_te)

# slice physio to utterance-level counts
#   we simply take the first N1 windows from physio_probs_tr as "train utterances",
#   next N2 windows as "dev utterances", and first N3 of physio_probs_te as test.
pp_tr = physio_probs_tr[:N1, 1]             # physio stress-prob on train
pp_dev= physio_probs_tr[N1:N1+N2, 1]        # physio stress-prob on dev
pp_te = physio_probs_te[:N3, 1]             # physio stress-prob on test

# ─── 3️⃣ BINARY STRESS VS NON-STRESS FUSION ──────────────────────────────
# map MELD 7-way → stress vs non-stress (fear,sadness,disgust,anger → stress)
stress_idxs = [2,3,5,6]
y_dev_bin = np.isin(y_dev, stress_idxs).astype(int)
y_te_bin  = np.isin(y_test, stress_idxs).astype(int)

# collapse to scalar stress probs
tp_dev = text_probs_dev[:, stress_idxs].sum(axis=1)
ap_dev = audio_probs_dev[:, stress_idxs].sum(axis=1)

tp_te  = text_probs_te[:, stress_idxs].sum(axis=1)
ap_te  = audio_probs_te[:, stress_idxs].sum(axis=1)

# ─── 4️⃣ GRID-SEARCH FUSION WEIGHTS on DEV ─────────────────────────────────
best = (0,0,0,-1)
grid = np.linspace(0,1,21)
for wt in grid:
    for wa in grid:
        wp = 1 - wt - wa
        if wp<0 or wp>1: continue
        fused_dev = wt*tp_dev + wa*ap_dev + wp*pp_dev
        preds_dev = (fused_dev>0.5).astype(int)
        f1 = f1_score(y_dev_bin, preds_dev)
        if f1>best[3]:
            best = (wt,wa,wp,f1)

wt, wa, wp, best_f1 = best
print(f"\n→ Best dev weights: text={wt:.2f}, audio={wa:.2f}, physio={wp:.2f} (dev F1={best_f1:.3f})")

# ─── 5️⃣ APPLY ON TEST & REPORT ───────────────────────────────────────────
fused_te = wt*tp_te + wa*ap_te + wp*pp_te
pred_te  = (fused_te>0.5).astype(int)

print("\n[Fused Stress vs Non-Stress on MELD Test]")
print("Accuracy:", accuracy_score(y_te_bin, pred_te))
print("F1-Score:", f1_score(y_te_bin, pred_te))
print(classification_report(y_te_bin, pred_te, target_names=["non-stress","stress"]))



=== Physio-Only RF on WESAD Test ===
              precision    recall  f1-score   support

    baseline       0.97      0.97      0.97      2641
      stress       0.97      0.98      0.97      1494
   amusement       0.94      0.89      0.92       837
  meditation       0.95      0.97      0.96      1772

    accuracy                           0.96      6744
   macro avg       0.96      0.95      0.95      6744
weighted avg       0.96      0.96      0.96      6744


▶️ MELD splits: train=2610, dev=1109, test=2610

=== Text-only LR on MELD Test ===
              precision    recall  f1-score   support

           0       0.42      0.13      0.19      1256
           1       0.12      0.10      0.11       281
           2       0.00      0.02      0.01        50
           3       0.08      0.16      0.11       208
           4       0.15      0.18      0.17       402
           5       0.03      0.18      0.06        68
           6       0.12      0.18      0.14       345

    accur

In [9]:
# INSTALL & IMPORTS 
!pip install --quiet scipy scikit-learn tqdm

import os, glob, pickle
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal            import butter, filtfilt, resample_poly, find_peaks
from sklearn.ensemble        import RandomForestClassifier, StackingClassifier
from sklearn.neural_network  import MLPClassifier
from sklearn.svm             import SVC
from sklearn.linear_model    import LogisticRegression
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics         import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')


#  BUILD & TRAIN WESAD PHYSIO MODEL
def convert_units(sig, mod):
    if mod=='EDA':    return ((sig/65536.0)*3.0)/0.12
    if mod=='ECG':    return ((sig/65536.0)-0.5)*3000.0
    if mod=='RESP':   return ((sig/65536.0)-0.5)*100.0
    return sig

def lowpass(sig, cutoff, fs):
    nyq = fs*0.5
    if cutoff>=nyq or sig.size<10:
        return sig
    b,a = butter(4, cutoff/nyq, btype='low')
    try:
        return filtfilt(b,a,sig)
    except:
        return sig

def downsample(sig, frm, to, mod=None):
    if mod: sig = convert_units(sig, mod)
    if frm==to: return sig
    f = lowpass(sig, to*0.5, frm)
    return resample_poly(f, to, frm)

def basic_stats(x):
    x = x.flatten()
    return [x.mean(), x.std(), x.min(), x.max(),
            np.percentile(x,25), np.percentile(x,50), np.percentile(x,75),
            x.ptp(), np.mean(np.abs(np.diff(x)))]

def ecg_feats(x, fs):
    x = x.flatten()
    peaks,_ = find_peaks(x, distance=fs*0.5)
    rr = np.diff(peaks)/fs if len(peaks)>1 else np.array([])
    hr  = 60/rr.mean() if rr.size>0 else 0
    hrv = rr.std()    if rr.size>0 else 0
    return [hr, hrv] + basic_stats(x)

def build_physio(root="/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"):
    fs_o, fs_e, fs_r = 700, 64, 4
    win, stride = fs_o*4, fs_o*2
    Xp, yp = [], []
    for pkl_fp in glob.glob(os.path.join(root, "S*", "S*.pkl")):
        with open(pkl_fp,'rb') as f:
            d = pickle.load(f, encoding='latin1')
        ecg, eda, resp = d['signal']['chest']['ECG'], d['signal']['chest']['EDA'], d['signal']['chest']['Resp']
        lbl700 = d['label']
        ecg_ds  = downsample(ecg, fs_o, fs_e, 'ECG')
        eda_ds  = downsample(eda, fs_o, fs_r, 'EDA')
        resp_ds = downsample(resp,fs_o, fs_r, 'RESP')
        for i in range(0, len(lbl700)-win+1, stride):
            lab,_ = Counter(lbl700[i:i+win]).most_common(1)[0]
            if lab not in (1,2,3,4): continue
            e0,e1 = int(i/fs_o*fs_e), int((i+win-1)/fs_o*fs_e)+1
            r0,r1 = int(i/fs_o*fs_r), int((i+win-1)/fs_o*fs_r)+1
            feats = ecg_feats(ecg_ds[e0:e1], fs_e) + basic_stats(eda_ds[r0:r1]) + basic_stats(resp_ds[r0:r1])
            Xp.append(feats); yp.append(lab-1)
    return np.array(Xp), np.array(yp)

X_physio, y_physio = build_physio()
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    X_physio, y_physio, test_size=0.3, stratify=y_physio, random_state=42
)

rf_physio = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42
).fit(Xp_tr, yp_tr)

print("\n[Physio-only RF]")
print(classification_report(yp_te, rf_physio.predict(Xp_te),
      target_names=['baseline','stress','amusement','meditation']))

physio_probs_tr = rf_physio.predict_proba(Xp_tr)
physio_probs_te = rf_physio.predict_proba(Xp_te)


# LOAD & PREPARE MELD TEXT/AUDIO FEATURES 
AUDIOTEXT = "/kaggle/input/audiotext"
train_df = pd.read_csv(f"{AUDIOTEXT}/train_sent_emo.csv")
dev_df   = pd.read_csv(f"{AUDIOTEXT}/dev_sent_emo.csv")
test_df  = pd.read_csv(f"{AUDIOTEXT}/test_sent_emo.csv")
N1, N2, N3 = len(train_df), len(dev_df), len(test_df)
print(f"\n[MELD splits] train={N1} dev={N2} test={N3}")

# labels (7-way)
if 'label' in train_df.columns:
    y_train = train_df['label'].values
    y_dev   = dev_df['label'].values
    y_test  = test_df['label'].values
else:
    emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
    y_train = train_df['Emotion'].map(emo_map).astype(int).values
    y_dev   = dev_df['Emotion'].map(emo_map).astype(int).values
    y_test  = test_df['Emotion'].map(emo_map).astype(int).values

# load pickles
with open(f"{AUDIOTEXT}/text_emotion.pkl",'rb')  as f: text_data  = pickle.load(f)
with open(f"{AUDIOTEXT}/audio_emotion.pkl",'rb') as f: audio_data = pickle.load(f)

def extract_meld(data, total):
    feats=[]
    for split in data:
        for _,arr in split.items():
            feats.append(arr)
    feats = np.vstack(feats)
    if feats.shape[0]>total: feats=feats[:total]
    if feats.shape[0]<total:
        pad = total-feats.shape[0]
        feats = np.vstack([feats, np.zeros((pad,feats.shape[1]))])
    return feats

X_text_all  = extract_meld(text_data,  N1+N2+N3)
X_audio_all = extract_meld(audio_data, N1+N2+N3)

X_text_tr, X_text_dev, X_text_te   = np.split(X_text_all,  [N1, N1+N2])
X_audio_tr, X_audio_dev, X_audio_te= np.split(X_audio_all, [N1, N1+N2])

# scale
sc_t = StandardScaler().fit(np.vstack([X_text_tr,X_text_dev]))
X_text_tr  = sc_t.transform(X_text_tr)
X_text_dev = sc_t.transform(X_text_dev)
X_text_te  = sc_t.transform(X_text_te)

sc_a = StandardScaler().fit(np.vstack([X_audio_tr,X_audio_dev]))
X_audio_tr  = sc_a.transform(X_audio_tr)
X_audio_dev = sc_a.transform(X_audio_dev)
X_audio_te  = sc_a.transform(X_audio_te)


# IMPROVED UNIMODAL TRAINING & SELECTION 
# TEXT
rf_text = RandomForestClassifier( n_estimators=300, class_weight='balanced', random_state=42 )
rf_text.fit(np.vstack([X_text_tr,X_text_dev]), np.hstack([y_train,y_dev]))
y_rf_text   = rf_text.predict(X_text_te)

mlp_text = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=200, random_state=42)
mlp_text.fit(np.vstack([X_text_tr,X_text_dev]), np.hstack([y_train,y_dev]))
y_mlp_text = mlp_text.predict(X_text_te)

stack_text = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
        ('rf', rf_text),
        ('mlp', mlp_text),
        ('svm', SVC(kernel='rbf', probability=True, class_weight='balanced'))
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=3
)
stack_text.fit(np.vstack([X_text_tr,X_text_dev]), np.hstack([y_train,y_dev]))
y_stack_text = stack_text.predict(X_text_te)

print("\n=== Text RF ===");   print(classification_report(y_test, y_rf_text))
print("=== Text MLP ===");      print(classification_report(y_test, y_mlp_text))
print("=== Text Stacked ==="); print(classification_report(y_test, y_stack_text))

text_models = {
    'rf':    (rf_text,  f1_score(y_test, y_rf_text,  average='weighted')),
    'mlp':   (mlp_text, f1_score(y_test, y_mlp_text, average='weighted')),
    'stack': (stack_text, f1_score(y_test, y_stack_text, average='weighted'))
}
best_text_name, (best_text_model, _) = max(text_models.items(), key=lambda kv: kv[1][1])
print(f"\n→ Best TEXT: {best_text_name}")

# AUDIO
rf_aud = RandomForestClassifier( n_estimators=300, class_weight='balanced', random_state=42 )
rf_aud.fit(np.vstack([X_audio_tr,X_audio_dev]), np.hstack([y_train,y_dev]))
y_rf_aud   = rf_aud.predict(X_audio_te)

mlp_aud = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=200, random_state=42)
mlp_aud.fit(np.vstack([X_audio_tr,X_audio_dev]), np.hstack([y_train,y_dev]))
y_mlp_aud = mlp_aud.predict(X_audio_te)

stack_aud = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
        ('rf', rf_aud),
        ('mlp', mlp_aud),
        ('svm', SVC(kernel='rbf', probability=True, class_weight='balanced'))
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=3
)
stack_aud.fit(np.vstack([X_audio_tr,X_audio_dev]), np.hstack([y_train,y_dev]))
y_stack_aud = stack_aud.predict(X_audio_te)

print("\n=== Audio RF ===");   print(classification_report(y_test, y_rf_aud))
print("=== Audio MLP ===");      print(classification_report(y_test, y_mlp_aud))
print("=== Audio Stacked ==="); print(classification_report(y_test, y_stack_aud))

audio_models = {
    'rf':    (rf_aud,  f1_score(y_test, y_rf_aud,  average='weighted')),
    'mlp':   (mlp_aud, f1_score(y_test, y_mlp_aud, average='weighted')),
    'stack': (stack_aud, f1_score(y_test, y_stack_aud, average='weighted'))
}
best_aud_name, (best_aud_model, _) = max(audio_models.items(), key=lambda kv: kv[1][1])
print(f"\n→ Best AUDIO: {best_aud_name}")


# LATE FUSION (Binary Stress vs Non-Stress)
# melt text/audio to stress probs
stress_idxs = [2,3,5,6]
tp_dev = best_text_model.predict_proba(X_text_dev)[:, stress_idxs].sum(axis=1)
ap_dev = best_aud_model .predict_proba(X_audio_dev)[:, stress_idxs].sum(axis=1)
pp_dev = physio_probs_tr[N1:N1+N2, 1]    # physio class=1 is stress

tp_te  = best_text_model.predict_proba(X_text_te)[:, stress_idxs].sum(axis=1)
ap_te  = best_aud_model .predict_proba(X_audio_te)[:, stress_idxs].sum(axis=1)
pp_te  = physio_probs_te[:N3, 1]

y_dev_bin = np.isin(np.hstack([y_dev]), stress_idxs).astype(int)
y_te_bin  = np.isin(y_test, stress_idxs).astype(int)

# grid-search weights
best = (0,0,0,-1)
grid = np.linspace(0,1,21)
for wt in grid:
    for wa in grid:
        wp = 1 - wt - wa
        if wp<0 or wp>1: continue
        fused = wt*tp_dev + wa*ap_dev + wp*pp_dev
        preds = (fused>0.5).astype(int)
        f1 = f1_score(y_dev_bin, preds)
        if f1>best[3]:
            best = (wt,wa,wp,f1)

wt, wa, wp, _ = best
print(f"\n→ Best fusion weights (dev): text={wt:.2f}, audio={wa:.2f}, physio={wp:.2f}")

# apply on test
fused_te = wt*tp_te + wa*ap_te + wp*pp_te
pred_te  = (fused_te>0.5).astype(int)

print("\n[Fused Stress vs Non-Stress Test]")
print("Accuracy:", accuracy_score(y_te_bin, pred_te))
print("F1-Score:", f1_score(y_te_bin, pred_te))
print(classification_report(y_te_bin, pred_te, target_names=["non-stress","stress"]))



[Physio-only RF]
              precision    recall  f1-score   support

    baseline       0.97      0.97      0.97      2641
      stress       0.97      0.98      0.97      1494
   amusement       0.94      0.89      0.92       837
  meditation       0.95      0.97      0.96      1772

    accuracy                           0.96      6744
   macro avg       0.96      0.95      0.95      6744
weighted avg       0.96      0.96      0.96      6744


[MELD splits] train=2610 dev=1109 test=2610

=== Text RF ===
              precision    recall  f1-score   support

           0       0.48      0.93      0.63      1256
           1       0.19      0.01      0.03       281
           2       0.00      0.00      0.00        50
           3       0.06      0.00      0.01       208
           4       0.15      0.02      0.04       402
           5       0.00      0.00      0.00        68
           6       0.11      0.02      0.03       345

    accuracy                           0.46      26

In [8]:
from sklearn.ensemble      import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import classification_report, accuracy_score

# ─── assume these are already defined ──────────────────────────────────────
# Xp_tr, Xp_te, yp_tr, yp_te           # from your WESAD split
# X_text_tr, X_text_dev, X_text_te     # from MELD text features
# X_audio_tr, X_audio_dev, X_audio_te  # from MELD audio features
# y_train (train labels), y_dev (dev labels), y_test (test labels)
# N1 = len(y_train), N2 = len(y_dev), N3 = len(y_test)

# ALIGN WESAD WINDOWS → utterance counts
#    map first N1 windows of Xp_tr to train, next N2 to dev, first N3 of Xp_te to test
Xp_uf_tr  = Xp_tr[        :N1 ]
Xp_uf_dev = Xp_tr[ N1 : N1+N2 ]
Xp_uf_te  = Xp_te[       :N3 ]

# sanity check shapes
assert Xp_uf_tr.shape[0]  == N1
assert Xp_uf_dev.shape[0] == N2
assert Xp_uf_te.shape[0]  == N3

#  BUILD EARLY‐FUSION MATRICES
X_early_tr  = np.hstack([Xp_uf_tr,  X_text_tr,  X_audio_tr])
X_early_dev = np.hstack([Xp_uf_dev, X_text_dev, X_audio_dev])
X_early_te  = np.hstack([Xp_uf_te,  X_text_te,  X_audio_te])

# combine train+dev for final training
X_train_all = np.vstack([X_early_tr, X_early_dev])
y_train_all = np.hstack([y_train,     y_dev])

# SCALE FEATURES
scaler = StandardScaler().fit(X_train_all)
X_train_all = scaler.transform(X_train_all)
X_early_te  = scaler.transform(X_early_te)

#  TRAIN A SINGLE CLASSIFIER (7‐way)
clf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42
)
clf.fit(X_train_all, y_train_all)

#  EVALUATE on MELD test
y_pred = clf.predict(X_early_te)
print("=== Early‐Fusion RF on MELD Test ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=[
    'neutral','surprise','fear','sadness','joy','disgust','anger'
]))


=== Early‐Fusion RF on MELD Test ===
Accuracy: 0.4793103448275862
              precision    recall  f1-score   support

     neutral       0.48      1.00      0.65      1256
    surprise       0.00      0.00      0.00       281
        fear       0.00      0.00      0.00        50
     sadness       0.00      0.00      0.00       208
         joy       0.00      0.00      0.00       402
     disgust       0.00      0.00      0.00        68
       anger       0.00      0.00      0.00       345

    accuracy                           0.48      2610
   macro avg       0.07      0.14      0.09      2610
weighted avg       0.23      0.48      0.31      2610



In [2]:
#only physio and text based
# ─── ASSUMES ───────────────────────────────────────────────────────────────
# • best_text_model: your chosen text classifier (e.g. the MLP or SVM)
# • X_text_dev, X_text_te, y_dev, y_test: MELD text splits
# • physio_probs_tr, physio_probs_te: RF-probabilities from WESAD
# • N1 = len(y_train), N2 = len(y_dev), N3 = len(y_test)

from sklearn.metrics import f1_score, accuracy_score, classification_report
import numpy as np

# 1️⃣ Compute each model’s “stress” probability on dev & test
stress_idxs = [2,3,5,6]  # MELD indices for fear, sadness, disgust, anger

# Text → sum over the four “stress” emotion probs
tp_dev = best_text_model.predict_proba(X_text_dev)[:, stress_idxs].sum(axis=1)
tp_te  = best_text_model.predict_proba(X_text_te) [:, stress_idxs].sum(axis=1)

# Physio → take the RF’s probability of its “stress” class (index 1)
pp_dev = physio_probs_tr[N1:N1+N2, 1]  # dev windows
pp_te  = physio_probs_te[:N3, 1]       # test windows

# 2️⃣ Prepare binary labels
y_dev_bin = np.isin(y_dev, stress_idxs).astype(int)
y_te_bin  = np.isin(y_test, stress_idxs).astype(int)

# 3️⃣ Grid-search the best weight for text (w_text) vs physio (w_physio=1−w_text)
best = (0,0.0)  # (w_text, best_f1)
grid = np.linspace(0,1,21)

for w in grid:
    fused_dev = w*tp_dev + (1-w)*pp_dev
    preds_dev = (fused_dev > 0.5).astype(int)
    f1 = f1_score(y_dev_bin, preds_dev)
    if f1 > best[1]:
        best = (w, f1)

w_text, best_f1 = best
w_physio = 1 - w_text
print(f"→ Best dev weights: text={w_text:.2f}, physio={w_physio:.2f} (F1={best_f1:.3f})")

# 4️⃣ Apply on test
fused_te = w_text*tp_te + w_physio*pp_te
pred_te  = (fused_te > 0.5).astype(int)

print("\n[Text + Physio Late-Fusion Test Results]")
print("Accuracy:", accuracy_score(y_te_bin, pred_te))
print("F1-Score:", f1_score(y_te_bin, pred_te))
print(classification_report(y_te_bin, pred_te, target_names=["non-stress","stress"]))


→ Best dev weights: text=0.00, physio=1.00 (F1=0.256)

[Text + Physio Late-Fusion Test Results]
Accuracy: 0.6455938697318008
F1-Score: 0.236168455821635
              precision    recall  f1-score   support

  non-stress       0.74      0.80      0.77      1939
      stress       0.26      0.21      0.24       671

    accuracy                           0.65      2610
   macro avg       0.50      0.50      0.50      2610
weighted avg       0.62      0.65      0.63      2610



In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics       import classification_report, accuracy_score
import numpy as np

# ─── ASSUMES ───────────────────────────────────────────────────────────────
# • best_text_model: your trained text classifier (MLP/SVM/etc.)
# • X_text_dev, X_text_te, y_dev, y_test: your MELD text splits
# • physio_probs_tr, physio_probs_te: RF-probs from WESAD
# • N1 = len(y_train), N2 = len(y_dev), N3 = len(y_test)

# 1️⃣ Collapse each to a single “stress” probability
stress_idxs = [2,3,5,6]  # MELD indices for fear, sadness, disgust, anger

# text
tp_dev = best_text_model.predict_proba(X_text_dev)[:, stress_idxs].sum(axis=1)
tp_te  = best_text_model.predict_proba(X_text_te )[:, stress_idxs].sum(axis=1)

# physio: class=1 is stress
pp_dev = physio_probs_tr[N1:N1+N2, 1]
pp_te  = physio_probs_te[:N3,       1]

# 2️⃣ Build meta-features and labels
X_meta_dev = np.vstack([tp_dev, pp_dev]).T    # shape (N2, 2)
y_meta_dev = np.isin(y_dev, stress_idxs).astype(int)

X_meta_te  = np.vstack([tp_te,  pp_te ]).T    # shape (N3, 2)
y_meta_te  = np.isin(y_test, stress_idxs).astype(int)

# 3️⃣ Train the tiny meta-learner
meta = LogisticRegression(class_weight='balanced', random_state=42)
meta.fit(X_meta_dev, y_meta_dev)

# inspect learned weights
print("Meta-learner weights (text, physio):", meta.coef_[0])

# 4️⃣ Evaluate on test
pred_meta = meta.predict(X_meta_te)
print("\n[Stacked Text+Physio Fusion]")
print("Accuracy:", accuracy_score(y_meta_te, pred_meta))
print(classification_report(y_meta_te, pred_meta, target_names=["non-stress","stress"]))


Meta-learner weights (text, physio): [1.60573845 0.07225075]

[Stacked Text+Physio Fusion]
Accuracy: 0.518007662835249
              precision    recall  f1-score   support

  non-stress       0.76      0.52      0.61      1939
      stress       0.27      0.53      0.36       671

    accuracy                           0.52      2610
   macro avg       0.52      0.52      0.49      2610
weighted avg       0.63      0.52      0.55      2610



In [5]:
from sklearn.ensemble      import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import classification_report, accuracy_score, f1_score
import numpy as np

# --- 1) Prepare binary labels for MELD splits ---
stress_idxs = [2,3,5,6]  # MELD indices of fear, sadness, disgust, anger
y_train_bin = np.isin(y_train, stress_idxs).astype(int)
y_dev_bin   = np.isin(y_dev,   stress_idxs).astype(int)
y_test_bin  = np.isin(y_test,  stress_idxs).astype(int)

# --- 2) Align WESAD windows to MELD utterance counts ---
Xp_tr_uf  = Xp_tr[       :N1 ]      # first N1 windows → train
Xp_dev_uf = Xp_tr[N1:N1+N2]        # next N2 windows → dev
Xp_te_uf  = Xp_te[      :N3 ]      # first N3 windows of test split

assert Xp_tr_uf.shape[0]  == N1
assert Xp_dev_uf.shape[0] == N2
assert Xp_te_uf.shape[0]  == N3

# --- 3) Early‐fusion feature matrices (text embeddings + physio feats) ---
X_train_ef = np.hstack([Xp_tr_uf,  X_text_tr ])
X_dev_ef   = np.hstack([Xp_dev_uf, X_text_dev])
X_test_ef  = np.hstack([Xp_te_uf,  X_text_te ])

# --- 4) Scale them ---
scaler = StandardScaler().fit(X_train_ef)
X_train_ef = scaler.transform(X_train_ef)
X_dev_ef   = scaler.transform(X_dev_ef)
X_test_ef  = scaler.transform(X_test_ef)

# --- 5) Train a binary RandomForest (you can swap in MLP or LR) ---
clf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42
)
clf.fit(
    np.vstack([ X_train_ef,  X_dev_ef ]),
    np.hstack([ y_train_bin, y_dev_bin ])
)

# --- 6) Evaluate on test ---
y_pred = clf.predict(X_test_ef)
print("=== Early‐Fusion Text+Physio (binary) ===")
print("Accuracy:", accuracy_score(y_test_bin, y_pred))
print("F1-Score:", f1_score(y_test_bin, y_pred))
print(classification_report(y_test_bin, y_pred, target_names=["non-stress","stress"]))


=== Early‐Fusion Text+Physio (binary) ===
Accuracy: 0.7386973180076628
F1-Score: 0.03125
              precision    recall  f1-score   support

  non-stress       0.74      0.99      0.85      1939
      stress       0.33      0.02      0.03       671

    accuracy                           0.74      2610
   macro avg       0.54      0.50      0.44      2610
weighted avg       0.64      0.74      0.64      2610



In [7]:
# ─── 0) INSTALL & IMPORTS ─────────────────────────────────────────────────
!pip install --quiet scipy scikit-learn tqdm

import os, glob, pickle
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal            import butter, filtfilt, resample_poly, find_peaks
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics         import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# ─── 1) BUILD & TRAIN PHYSIO MODEL ────────────────────────────────────────
def convert_units(sig, mod):
    if   mod=='EDA':    return ((sig/65536.0)*3.0)/0.12
    if   mod=='ECG':    return ((sig/65536.0)-0.5)*3000.0
    if   mod=='RESP':   return ((sig/65536.0)-0.5)*100.0
    return sig

def lowpass(sig, cutoff, fs):
    nyq = fs*0.5
    if cutoff>=nyq or sig.size<10:
        return sig
    b,a = butter(4, cutoff/nyq, btype='low')
    try:   return filtfilt(b,a,sig)
    except: return sig

def downsample(sig, frm, to, mod=None):
    if mod: sig = convert_units(sig, mod)
    if frm==to: return sig
    f = lowpass(sig, to*0.5, frm)
    return resample_poly(f, to, frm)

def basic_stats(x):
    x = x.flatten()
    return [
        x.mean(), x.std(), x.min(), x.max(),
        np.percentile(x,25), np.percentile(x,50), np.percentile(x,75),
        x.ptp(), np.mean(np.abs(np.diff(x)))
    ]

def ecg_feats(x, fs):
    x = x.flatten()
    peaks,_ = find_peaks(x, distance=fs*0.5)
    rr = np.diff(peaks)/fs if len(peaks)>1 else np.array([])
    hr  = 60/rr.mean() if rr.size>0 else 0
    hrv = rr.std()    if rr.size>0 else 0
    return [hr, hrv] + basic_stats(x)

def build_physio(path):
    fs_o, fs_e, fs_r = 700, 64, 4
    win, stride = fs_o*4, fs_o*2
    X, y = [], []
    for fp in glob.glob(os.path.join(path,"S*","S*.pkl")):
        with open(fp,'rb') as f:
            d = pickle.load(f, encoding='latin1')
        ecg, eda, resp = d['signal']['chest']['ECG'], d['signal']['chest']['EDA'], d['signal']['chest']['Resp']
        lbl = d['label']
        ecg_ds  = downsample(ecg, fs_o, fs_e, 'ECG')
        eda_ds  = downsample(eda, fs_o, fs_r, 'EDA')
        resp_ds = downsample(resp,fs_o, fs_r,'RESP')
        for i in range(0, len(lbl)-win+1, stride):
            lab,_ = Counter(lbl[i:i+win]).most_common(1)[0]
            if lab not in (1,2,3,4): continue
            e0,e1 = int(i/fs_o*fs_e), int((i+win-1)/fs_o*fs_e)+1
            r0,r1 = int(i/fs_o*fs_r), int((i+win-1)/fs_o*fs_r)+1
            feats = ecg_feats(ecg_ds[e0:e1], fs_e) + basic_stats(eda_ds[r0:r1]) + basic_stats(resp_ds[r0:r1])
            X.append(feats); y.append(lab-1)
    return np.array(X), np.array(y)

# Build & split
X_physio, y_physio = build_physio("/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD")
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(X_physio, y_physio, test_size=0.3,
                                              stratify=y_physio, random_state=42)

# Train RF
rf_physio = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_physio.fit(Xp_tr, yp_tr)
physio_probs_tr = rf_physio.predict_proba(Xp_tr)
physio_probs_te = rf_physio.predict_proba(Xp_te)

print("\n[Physio-only RF]")
print(classification_report(yp_te, rf_physio.predict(Xp_te),
      target_names=['baseline','stress','amusement','meditation']))

# ─── 2) LOAD & PREP TEXT FEATURES ─────────────────────────────────────────
AUDIOTEXT = "/kaggle/input/audiotext"
train_df = pd.read_csv(f"{AUDIOTEXT}/train_sent_emo.csv")
dev_df   = pd.read_csv(f"{AUDIOTEXT}/dev_sent_emo.csv")
test_df  = pd.read_csv(f"{AUDIOTEXT}/test_sent_emo.csv")
N1, N2, N3 = len(train_df), len(dev_df), len(test_df)

# Map to 0–6
emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
y_train = train_df.get('label', train_df['Emotion'].map(emo_map)).values
y_dev   = dev_df.get('label',   dev_df['Emotion'].map(emo_map)).values
y_test  = test_df.get('label',  test_df['Emotion'].map(emo_map)).values

# Load embeddings
with open(f"{AUDIOTEXT}/text_emotion.pkl",'rb')  as f: text_data  = pickle.load(f)
def extract_all(data, total):
    rows=[]
    for split in data:
        for _,arr in split.items():
            rows.append(arr)
    arr = np.vstack(rows)
    if arr.shape[0]>total: arr=arr[:total]
    if arr.shape[0]<total:
        pad = total-arr.shape[0]
        arr = np.vstack([arr, np.zeros((pad,arr.shape[1]))])
    return arr

X_text_all = extract_all(text_data, N1+N2+N3)
X_text_tr, X_text_dev, X_text_te = np.split(X_text_all, [N1, N1+N2])

# Scale
sc = StandardScaler().fit(np.vstack([X_text_tr,X_text_dev]))
X_text_tr  = sc.transform(X_text_tr)
X_text_dev = sc.transform(X_text_dev)
X_text_te  = sc.transform(X_text_te)

# ─── 3) TRAIN TEXT SVM ────────────────────────────────────────────────────
svm_text = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
svm_text.fit(np.vstack([X_text_tr, X_text_dev]), np.hstack([y_train, y_dev]))

print("\n[Text-only SVM on MELD Test]")
print(classification_report(y_test, svm_text.predict(X_text_te)))

# ─── 4) LATE-FUSION TEXT + PHYSIO ────────────────────────────────────────
# Stress indices in MELD
stress_idxs = [2,3,5,6]

# Text stress-prob
tp_dev = svm_text.predict_proba(X_text_dev)[:, stress_idxs].sum(axis=1)
tp_te  = svm_text.predict_proba(X_text_te) [:, stress_idxs].sum(axis=1)
# Physio stress-prob
pp_dev = physio_probs_tr[N1:N1+N2, 1]
pp_te  = physio_probs_te[:N3,       1]

# Binary labels
y_dev_bin = np.isin(y_dev, stress_idxs).astype(int)
y_te_bin  = np.isin(y_test, stress_idxs).astype(int)

# Grid-search fusion weight
best = (0,0.0)
for w in np.linspace(0,1,21):
    fused = w*tp_dev + (1-w)*pp_dev
    preds = (fused>0.5).astype(int)
    f1 = f1_score(y_dev_bin, preds)
    if f1>best[1]:
        best = (w, f1)
w_text, best_f1 = best
w_physio = 1 - w_text
print(f"\n→ Dev best weights: text={w_text:.2f}, physio={w_physio:.2f}, F1={best_f1:.3f}")

# Apply on test
fused_te = w_text*tp_te + w_physio*pp_te
pred_te  = (fused_te>0.5).astype(int)

print("\n[LATE-FUSION Text+Physio Test]")
print("Accuracy:", accuracy_score(y_te_bin, pred_te))
print("F1-Score:", f1_score(y_te_bin, pred_te))
print(classification_report(y_te_bin, pred_te, target_names=["non-stress","stress"]))



[Physio-only RF]
              precision    recall  f1-score   support

    baseline       0.97      0.97      0.97      2641
      stress       0.97      0.98      0.97      1494
   amusement       0.94      0.89      0.92       837
  meditation       0.95      0.97      0.96      1772

    accuracy                           0.96      6744
   macro avg       0.96      0.95      0.95      6744
weighted avg       0.96      0.96      0.96      6744


[Text-only SVM on MELD Test]
              precision    recall  f1-score   support

           0       0.44      0.07      0.12      1256
           1       0.09      0.18      0.12       281
           2       0.00      0.00      0.00        50
           3       0.09      0.13      0.11       208
           4       0.16      0.15      0.16       402
           5       0.03      0.10      0.04        68
           6       0.14      0.30      0.19       345

    accuracy                           0.13      2610
   macro avg       0.14      

In [8]:
from sklearn.svm             import SVC
from sklearn.model_selection import RandomizedSearchCV
from sklearn.calibration     import CalibratedClassifierCV
from sklearn.linear_model    import LogisticRegression
from sklearn.metrics         import f1_score, accuracy_score, classification_report
import numpy as np

# ❶ — TUNE the Text SVM on train+dev for MELD’s 7-way task
param_dist = {
    "C":     [0.1, 1, 10, 100],
    "gamma": ["scale","auto", 0.01, 0.1, 1]
}
svm_base = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
search = RandomizedSearchCV(
    svm_base, param_dist, n_iter=10, cv=3,
    scoring="f1_weighted", n_jobs=-1, random_state=42
)
search.fit(np.vstack([X_text_tr, X_text_dev]), np.hstack([y_train, y_dev]))
best_svm = search.best_estimator_
print("👉 Best SVM params:", search.best_params_)

# ❷ — CALIBRATE its probabilities on the DEV split
cal_svm = CalibratedClassifierCV(best_svm, cv='prefit')
cal_svm.fit(X_text_dev, y_dev)

# You now have better‐calibrated text stress‐scores:
# stress_idxs = [2,3,5,6]
tp_dev = cal_svm.predict_proba(X_text_dev)[:, stress_idxs].sum(axis=1)
tp_te  = cal_svm.predict_proba(X_text_te )[:, stress_idxs].sum(axis=1)

# ❸ — EXTRACT Physio stress‐scores
pp_dev = physio_probs_tr[N1:N1+N2, 1]
pp_te  = physio_probs_te[:N3,       1]

# ❹ — BUILD meta‐features for DEV & TEST
X_meta_dev = np.vstack([tp_dev, pp_dev]).T   # shape (dev_n, 2)
y_meta_dev = np.isin(y_dev, stress_idxs).astype(int)
X_meta_te  = np.vstack([tp_te,  pp_te ]).T
y_meta_te  = np.isin(y_test, stress_idxs).astype(int)

# ❺ — TRAIN a tiny Logistic‐regression coach on DEV
meta = LogisticRegression(class_weight='balanced', random_state=42)
meta.fit(X_meta_dev, y_meta_dev)
print("Meta weights (text, physio):", meta.coef_[0])

# ❻ — FIND optimal decision threshold on DEV
probs_dev = meta.predict_proba(X_meta_dev)[:,1]
best_thr, best_f1 = 0.5, 0
for thr in np.linspace(0.1,0.9,81):
    preds = (probs_dev > thr).astype(int)
    f1 = f1_score(y_meta_dev, preds)
    if f1>best_f1:
        best_f1, best_thr = f1, thr
print(f"→ Best meta-threshold on dev: {best_thr:.2f} (F1={best_f1:.3f})")

# ❼ — APPLY on TEST
probs_te = meta.predict_proba(X_meta_te)[:,1]
preds_te = (probs_te > best_thr).astype(int)

print("\n[Final Text+Physio Late-Fusion with Calibrated SVM & Meta-Learner]")
print("Accuracy:", accuracy_score(y_meta_te, preds_te))
print("F1-Score:", f1_score(y_meta_te, preds_te))
print(classification_report(y_meta_te, preds_te, target_names=["non-stress","stress"]))


👉 Best SVM params: {'gamma': 0.1, 'C': 100}
Meta weights (text, physio): [6.28810654 0.10708973]
→ Best meta-threshold on dev: 0.54 (F1=0.815)

[Final Text+Physio Late-Fusion with Calibrated SVM & Meta-Learner]
Accuracy: 0.6498084291187739
F1-Score: 0.23193277310924368
              precision    recall  f1-score   support

  non-stress       0.75      0.80      0.77      1939
      stress       0.27      0.21      0.23       671

    accuracy                           0.65      2610
   macro avg       0.51      0.50      0.50      2610
weighted avg       0.62      0.65      0.63      2610



In [5]:
# ─── 0️⃣ INSTALL & IMPORTS ─────────────────────────────────────────────────
!pip install --quiet scipy scikit-learn tqdm

import glob, pickle, numpy as np, pandas as pd
from collections import Counter
from scipy.signal            import butter, filtfilt, resample_poly, find_peaks
from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.impute          import SimpleImputer
from sklearn.metrics         import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# ─── 1️⃣ LOAD & PREP TEXT EMBEDDINGS ───────────────────────────────────────
DATA_DIR = "/kaggle/input/audiotext"
train_df = pd.read_csv(f"{DATA_DIR}/train_sent_emo.csv")
dev_df   = pd.read_csv(f"{DATA_DIR}/dev_sent_emo.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test_sent_emo.csv")
N1, N2, N3 = len(train_df), len(dev_df), len(test_df)

with open(f"{DATA_DIR}/text_emotion.pkl",'rb') as f:
    text_data = pickle.load(f)

def extract_all(data, total):
    rows = []
    for split in data:
        for _, arr in split.items():
            rows.append(arr)
    arr = np.vstack(rows)
    if arr.shape[0] > total: arr = arr[:total]
    if arr.shape[0] < total:
        pad = total - arr.shape[0]
        arr = np.vstack([arr, np.zeros((pad, arr.shape[1]))])
    return arr

X_text_all  = extract_all(text_data, N1+N2+N3)
X_text_tr, X_text_dev, X_text_te = np.split(X_text_all, [N1, N1+N2])

# map MELD labels to 0…6
emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
if 'label' in train_df.columns:
    y_train = train_df['label'].values
    y_dev   = dev_df  ['label'].values
    y_test  = test_df ['label'].values
else:
    y_train = train_df['Emotion'].map(emo_map).fillna(0).astype(int).values
    y_dev   = dev_df  ['Emotion'].map(emo_map).fillna(0).astype(int).values
    y_test  = test_df ['Emotion'].map(emo_map).fillna(0).astype(int).values

# ─── 2️⃣ BUILD & PREP PHYSIO FEATURES ──────────────────────────────────────
def build_physio(path):
    fs_o, fs_e, fs_r = 700, 64, 4
    win, stride = fs_o*4, fs_o*2
    X, y = [], []
    for fp in glob.glob(path+"/S*/S*.pkl"):
        with open(fp,'rb') as f:
            d = pickle.load(f, encoding='latin1')
        ecg, eda, resp = d['signal']['chest']['ECG'], d['signal']['chest']['EDA'], d['signal']['chest']['Resp']
        lbl700 = d['label']
        # converters & downsampling
        def conv(sig, mod):
            if mod=='ECG':  return ((sig/65536.0)-0.5)*3000.0
            if mod=='EDA':  return ((sig/65536.0)*3.0)/0.12
            if mod=='RESP': return ((sig/65536.0)-0.5)*100.0
            return sig
        def down(sig, frm, to, mod):
            s = conv(sig, mod)
            return resample_poly(s, to, frm)
        ecg_ds  = down(ecg,  fs_o, fs_e, 'ECG')
        eda_ds  = down(eda,  fs_o, fs_r, 'EDA')
        resp_ds = down(resp, fs_o, fs_r, 'RESP')
        # sliding windows
        for i in range(0, len(lbl700)-win+1, stride):
            lab,_ = Counter(lbl700[i:i+win]).most_common(1)[0]
            if lab not in (1,2,3,4): continue
            e0,e1 = int(i/fs_o*fs_e), int((i+win-1)/fs_o*fs_e)+1
            r0,r1 = int(i/fs_o*fs_r), int((i+win-1)/fs_o*fs_r)+1
            def stats(a):
                if len(a)==0: return [0]*9
                return [a.mean(), a.std(), a.min(), a.max(),
                        np.percentile(a,25), np.percentile(a,50),
                        np.percentile(a,75), a.ptp(),
                        np.mean(np.abs(np.diff(a)))]
            X.append(stats(ecg_ds[e0:e1]) + stats(eda_ds[r0:r1]) + stats(resp_ds[r0:r1]))
            y.append(lab-1)
    return np.array(X, dtype=np.float32), np.array(y, dtype=int)

PHYSIO_BASE = "/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"
X_physio, y_physio = build_physio(PHYSIO_BASE)
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    X_physio, y_physio, test_size=0.3, stratify=y_physio, random_state=42
)

# align windows → MELD splits
Xp_tr_uf  = Xp_tr[       :N1 ]
Xp_dev_uf = Xp_tr[N1:N1+N2]
Xp_te_uf  = Xp_te[      :N3 ]

# ─── 3️⃣ EARLY-FUSION FEATURE MATRICES ────────────────────────────────────
X_train_ef = np.hstack([Xp_tr_uf,  X_text_tr ])
X_dev_ef   = np.hstack([Xp_dev_uf, X_text_dev])
X_test_ef  = np.hstack([Xp_te_uf,  X_text_te ])

# ─── 4️⃣ IMPUTE MISSING & SCALE ───────────────────────────────────────────
imp = SimpleImputer(strategy='mean')
X_train_ef = imp.fit_transform(X_train_ef)
X_dev_ef   = imp.transform(X_dev_ef)
X_test_ef  = imp.transform(X_test_ef)

scaler = StandardScaler().fit(X_train_ef)
X_train_ef = scaler.transform(X_train_ef)
X_dev_ef   = scaler.transform(X_dev_ef)
X_test_ef  = scaler.transform(X_test_ef)

# ─── 5️⃣ TRAIN & EVALUATE BINARY STRESS vs NON-STRESS ─────────────────────
stress_idxs = [2,3,5,6]
y_train_bin = np.isin(y_train, stress_idxs).astype(int)
y_dev_bin   = np.isin(y_dev,   stress_idxs).astype(int)
y_test_bin  = np.isin(y_test,  stress_idxs).astype(int)

clf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
clf.fit(np.vstack([X_train_ef, X_dev_ef]), np.hstack([y_train_bin, y_dev_bin]))

pred = clf.predict(X_test_ef)
print("=== Early-Fusion Text+Physio (binary) ===")
print("Accuracy:", accuracy_score(y_test_bin, pred))
print("F1-Score:",    f1_score(y_test_bin, pred))
print(classification_report(y_test_bin, pred, target_names=["non-stress","stress"]))


/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


=== Early-Fusion Text+Physio (binary) ===
Accuracy: 0.7333333333333333
F1-Score: 0.02793296089385475
              precision    recall  f1-score   support

  non-stress       0.74      0.98      0.85      1939
      stress       0.22      0.01      0.03       671

    accuracy                           0.73      2610
   macro avg       0.48      0.50      0.44      2610
weighted avg       0.61      0.73      0.64      2610



In [1]:
# ─── Late‐Fusion Text + Physio (Fixed Alignment) ───────────────────────────

# 0️⃣ INSTALL & IMPORTS
!pip install --quiet scipy scikit-learn tqdm

import glob, pickle, numpy as np, pandas as pd
from collections import Counter
from scipy.signal           import resample_poly
from sklearn.impute         import SimpleImputer
from sklearn.ensemble       import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm            import SVC
from sklearn.linear_model   import LogisticRegression
from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils          import resample
from sklearn.metrics        import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# ─── 1️⃣ LOAD & PREP TEXT EMBEDDINGS ───────────────────────────────────────
TXT_DIR = "/kaggle/input/audiotext"
train_df = pd.read_csv(f"{TXT_DIR}/train_sent_emo.csv")
dev_df   = pd.read_csv(f"{TXT_DIR}/dev_sent_emo.csv")
test_df  = pd.read_csv(f"{TXT_DIR}/test_sent_emo.csv")
N1, N2, N3 = len(train_df), len(dev_df), len(test_df)

with open(f"{TXT_DIR}/text_emotion.pkl",'rb') as f:
    text_data = pickle.load(f)

def extract_all(data, total):
    rows = []
    for split in data:
        for _, arr in split.items():
            rows.append(arr)
    arr = np.vstack(rows)
    if arr.shape[0] > total: arr = arr[:total]
    if arr.shape[0] < total:
        pad = total - arr.shape[0]
        arr = np.vstack([arr, np.zeros((pad, arr.shape[1]))])
    return arr

X_text_all  = extract_all(text_data, N1+N2+N3)
X_text_tr, X_text_dev, X_text_te = np.split(X_text_all, [N1, N1+N2])

emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
if 'label' in train_df.columns:
    y_train = train_df['label'].values
    y_dev   = dev_df  ['label'].values
    y_test  = test_df ['label'].values
else:
    y_train = train_df['Emotion'].map(emo_map).fillna(0).astype(int).values
    y_dev   = dev_df  ['Emotion'].map(emo_map).fillna(0).astype(int).values
    y_test  = test_df ['Emotion'].map(emo_map).fillna(0).astype(int).values

# ─── 2️⃣ TRAIN & SELECT BEST TEXT MODEL ────────────────────────────────────
# a) RF
rf_text = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
rf_text.fit(np.vstack([X_text_tr, X_text_dev]), np.hstack([y_train, y_dev]))
f1_rf = f1_score(y_dev, rf_text.predict(X_text_dev), average='weighted')

# b) MLP
mlp_text = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=200, random_state=42)
mlp_text.fit(np.vstack([X_text_tr, X_text_dev]), np.hstack([y_train, y_dev]))
f1_mlp = f1_score(y_dev, mlp_text.predict(X_text_dev), average='weighted')

# c) Stacked
stack_text = StackingClassifier(
    estimators=[
        ('rf', rf_text),
        ('mlp', mlp_text),
        ('svm', SVC(kernel='rbf', probability=True, class_weight='balanced'))
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=3
)
stack_text.fit(np.vstack([X_text_tr, X_text_dev]), np.hstack([y_train, y_dev]))
f1_stack = f1_score(y_dev, stack_text.predict(X_text_dev), average='weighted')

# pick best
text_models = {'rf':(rf_text,f1_rf), 'mlp':(mlp_text,f1_mlp), 'stack':(stack_text,f1_stack)}
best_text_name, (best_text_model, _) = max(text_models.items(), key=lambda kv: kv[1][1])
print(f"→ Best TEXT model: {best_text_name} (dev F1={max(f1_rf,f1_mlp,f1_stack):.3f})")

# ─── 3️⃣ BUILD & TRAIN PHYSIO RF ───────────────────────────────────────────
def build_physio(path):
    fs_o,fs_e,fs_r = 700,64,4
    win,stride = fs_o*4,fs_o*2
    X,y = [],[]
    for fp in glob.glob(path+"/S*/S*.pkl"):
        with open(fp,'rb') as f: d=pickle.load(f,encoding='latin1')
        ecg,eda,resp,lbl = d['signal']['chest']['ECG'],d['signal']['chest']['EDA'],d['signal']['chest']['Resp'],d['label']
        def conv(sig,mod):
            if mod=='ECG':  return ((sig/65536.0)-0.5)*3000.0
            if mod=='EDA':  return ((sig/65536.0)*3.0)/0.12
            if mod=='RESP': return ((sig/65536.0)-0.5)*100.0
            return sig
        def down(sig, to, mod):
            return resample_poly(conv(sig,mod), to, fs_o)
        ecg_ds, eda_ds, resp_ds = down(ecg,fs_e,'ECG'), down(eda,fs_r,'EDA'), down(resp,fs_r,'RESP')
        for i in range(0,len(lbl)-win+1,stride):
            lab,_ = Counter(lbl[i:i+win]).most_common(1)[0]
            if lab not in (1,2,3,4): continue
            e0,e1=int(i/fs_o*fs_e),int((i+win-1)/fs_o*fs_e)+1
            r0,r1=int(i/fs_o*fs_r),int((i+win-1)/fs_o*fs_r)+1
            def stats(a):
                if a.size==0: return [0]*9
                return [a.mean(),a.std(),a.min(),a.max(),
                        np.percentile(a,25),np.percentile(a,50),np.percentile(a,75),
                        a.ptp(), np.mean(np.abs(np.diff(a)))]
            X.append(stats(ecg_ds[e0:e1])+stats(eda_ds[r0:r1])+stats(resp_ds[r0:r1]))
            y.append(lab-1)
    return np.array(X,float), np.array(y,int)

PHYSIO_DIR = "/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"
X_physio_raw,y_physio = build_physio(PHYSIO_DIR)
imp = SimpleImputer(strategy='mean')
X_physio = imp.fit_transform(X_physio_raw)

Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    X_physio, y_physio, test_size=0.3, stratify=y_physio, random_state=42
)
rf_physio = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_physio.fit(Xp_tr, yp_tr)

# align physio windows to MELD splits
Xp_tr_uf  = Xp_tr[      :N1 ]
Xp_dev_uf = Xp_tr[N1:N1+N2]
Xp_te_uf  = Xp_te[     :N3 ]

# ─── 4️⃣ COMPUTE “STRESS” SCORES ───────────────────────────────────────────
stress_idxs = [2,3,5,6]

tp_tr  = best_text_model.predict_proba(X_text_tr)  [:, stress_idxs].sum(1)
tp_dev = best_text_model.predict_proba(X_text_dev) [:, stress_idxs].sum(1)
tp_te  = best_text_model.predict_proba(X_text_te)  [:, stress_idxs].sum(1)

pp_tr  = rf_physio.predict_proba(Xp_tr_uf)[:,1]
pp_dev = rf_physio.predict_proba(Xp_dev_uf)[:,1]
pp_te2 = rf_physio.predict_proba(Xp_te_uf)[:,1]

y_train_bin = np.isin(y_train, stress_idxs).astype(int)
y_dev_bin   = np.isin(y_dev,   stress_idxs).astype(int)
y_test_bin  = np.isin(y_test,  stress_idxs).astype(int)

# ─── 5️⃣ TRAIN META‐LEARNER ON TRAIN ONLY & TUNE ON DEV ───────────────────
X_meta_tr   = np.vstack([tp_tr,  pp_tr ]).T
X_meta_dev  = np.vstack([tp_dev, pp_dev]).T

meta = LogisticRegression(class_weight='balanced', random_state=42)
meta.fit(X_meta_tr, y_train_bin)

# threshold tune on dev
probs_dev,best_thr,best_f1 = meta.predict_proba(X_meta_dev)[:,1], 0.5, 0
for thr in np.linspace(0.1,0.9,81):
    p = (probs_dev>thr).astype(int)
    f1 = f1_score(y_dev_bin,p)
    if f1>best_f1:
        best_f1,best_thr = f1,thr
print(f"Meta threshold on dev: {best_thr:.2f} (F1={best_f1:.3f})")

# EVALUATE ON TEST
X_meta_te = np.vstack([tp_te, pp_te2]).T
pred_te   = (meta.predict_proba(X_meta_te)[:,1] > best_thr).astype(int)

print("\n[Late-Fusion Meta‐Learner – Test]")
print("Accuracy:", accuracy_score(y_test_bin, pred_te))
print("F1-Score:", f1_score(y_test_bin, pred_te))
print(classification_report(y_test_bin, pred_te, target_names=["non-stress","stress"]))


→ Best TEXT model: rf (dev F1=1.000)
→ Meta threshold on dev: 0.15 (F1=1.000)

[Late-Fusion Meta‐Learner – Test]
Accuracy: 0.5226053639846743
F1-Score: 0.3414376321353066
              precision    recall  f1-score   support

  non-stress       0.75      0.54      0.63      1939
      stress       0.26      0.48      0.34       671

    accuracy                           0.52      2610
   macro avg       0.51      0.51      0.48      2610
weighted avg       0.62      0.52      0.55      2610



In [1]:
# ─── Late‐Fusion Text + Physio (Fixed Alignment) ───────────────────────────

# 0️⃣ INSTALL & IMPORTS
!pip install --quiet scipy scikit-learn tqdm

import glob, pickle, numpy as np, pandas as pd
from collections import Counter
from scipy.signal           import resample_poly
from sklearn.impute         import SimpleImputer
from sklearn.ensemble       import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm            import SVC
from sklearn.linear_model   import LogisticRegression
from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils          import resample
from sklearn.metrics        import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# ─── 1️⃣ LOAD & PREP TEXT EMBEDDINGS ───────────────────────────────────────
TXT_DIR = "/kaggle/input/audiotext"
train_df = pd.read_csv(f"{TXT_DIR}/train_sent_emo.csv")
dev_df   = pd.read_csv(f"{TXT_DIR}/dev_sent_emo.csv")
test_df  = pd.read_csv(f"{TXT_DIR}/test_sent_emo.csv")
N1, N2, N3 = len(train_df), len(dev_df), len(test_df)

with open(f"{TXT_DIR}/text_emotion.pkl",'rb') as f:
    text_data = pickle.load(f)

def extract_all(data, total):
    rows = []
    for split in data:
        for _, arr in split.items():
            rows.append(arr)
    arr = np.vstack(rows)
    if arr.shape[0] > total: arr = arr[:total]
    if arr.shape[0] < total:
        pad = total - arr.shape[0]
        arr = np.vstack([arr, np.zeros((pad, arr.shape[1]))])
    return arr

X_text_all  = extract_all(text_data, N1+N2+N3)
X_text_tr, X_text_dev, X_text_te = np.split(X_text_all, [N1, N1+N2])

emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
if 'label' in train_df.columns:
    y_train = train_df['label'].values
    y_dev   = dev_df  ['label'].values
    y_test  = test_df ['label'].values
else:
    y_train = train_df['Emotion'].map(emo_map).fillna(0).astype(int).values
    y_dev   = dev_df  ['Emotion'].map(emo_map).fillna(0).astype(int).values
    y_test  = test_df ['Emotion'].map(emo_map).fillna(0).astype(int).values

# ─── 2️⃣ TRAIN & SELECT BEST TEXT MODEL ────────────────────────────────────
# a) RF
rf_text = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
rf_text.fit(np.vstack([X_text_tr, X_text_dev]), np.hstack([y_train, y_dev]))
f1_rf = f1_score(y_dev, rf_text.predict(X_text_dev), average='weighted')

# b) MLP
mlp_text = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=200, random_state=42)
mlp_text.fit(np.vstack([X_text_tr, X_text_dev]), np.hstack([y_train, y_dev]))
f1_mlp = f1_score(y_dev, mlp_text.predict(X_text_dev), average='weighted')

# c) Stacked
stack_text = StackingClassifier(
    estimators=[
        ('rf', rf_text),
        ('mlp', mlp_text),
        ('svm', SVC(kernel='rbf', probability=True, class_weight='balanced'))
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=3
)
stack_text.fit(np.vstack([X_text_tr, X_text_dev]), np.hstack([y_train, y_dev]))
f1_stack = f1_score(y_dev, stack_text.predict(X_text_dev), average='weighted')

# pick best
text_models = {'rf':(rf_text,f1_rf), 'mlp':(mlp_text,f1_mlp), 'stack':(stack_text,f1_stack)}
best_text_name, (best_text_model, _) = max(text_models.items(), key=lambda kv: kv[1][1])
print(f"→ Best TEXT model: {best_text_name} (dev F1={max(f1_rf,f1_mlp,f1_stack):.3f})")

# ─── 3️⃣ BUILD & TRAIN PHYSIO RF ───────────────────────────────────────────
def build_physio(path):
    fs_o,fs_e,fs_r = 700,64,4
    win,stride = fs_o*4,fs_o*2
    X,y = [],[]
    for fp in glob.glob(path+"/S*/S*.pkl"):
        with open(fp,'rb') as f: d=pickle.load(f,encoding='latin1')
        ecg,eda,resp,lbl = d['signal']['chest']['ECG'],d['signal']['chest']['EDA'],d['signal']['chest']['Resp'],d['label']
        def conv(sig,mod):
            if mod=='ECG':  return ((sig/65536.0)-0.5)*3000.0
            if mod=='EDA':  return ((sig/65536.0)*3.0)/0.12
            if mod=='RESP': return ((sig/65536.0)-0.5)*100.0
            return sig
        def down(sig, to, mod):
            return resample_poly(conv(sig,mod), to, fs_o)
        ecg_ds, eda_ds, resp_ds = down(ecg,fs_e,'ECG'), down(eda,fs_r,'EDA'), down(resp,fs_r,'RESP')
        for i in range(0,len(lbl)-win+1,stride):
            lab,_ = Counter(lbl[i:i+win]).most_common(1)[0]
            if lab not in (1,2,3,4): continue
            e0,e1=int(i/fs_o*fs_e),int((i+win-1)/fs_o*fs_e)+1
            r0,r1=int(i/fs_o*fs_r),int((i+win-1)/fs_o*fs_r)+1
            def stats(a):
                if a.size==0: return [0]*9
                return [a.mean(),a.std(),a.min(),a.max(),
                        np.percentile(a,25),np.percentile(a,50),np.percentile(a,75),
                        a.ptp(), np.mean(np.abs(np.diff(a)))]
            X.append(stats(ecg_ds[e0:e1])+stats(eda_ds[r0:r1])+stats(resp_ds[r0:r1]))
            y.append(lab-1)
    return np.array(X,float), np.array(y,int)

PHYSIO_DIR = "/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"
X_physio_raw,y_physio = build_physio(PHYSIO_DIR)
imp = SimpleImputer(strategy='mean')
X_physio = imp.fit_transform(X_physio_raw)

Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    X_physio, y_physio, test_size=0.3, stratify=y_physio, random_state=42
)
rf_physio = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_physio.fit(Xp_tr, yp_tr)

# align physio windows to MELD splits
Xp_tr_uf  = Xp_tr[      :N1 ]
Xp_dev_uf = Xp_tr[N1:N1+N2]
Xp_te_uf  = Xp_te[     :N3 ]

# ─── 4️⃣ COMPUTE “STRESS” SCORES ───────────────────────────────────────────
stress_idxs = [2,3,5,6]

tp_tr  = best_text_model.predict_proba(X_text_tr)  [:, stress_idxs].sum(1)
tp_dev = best_text_model.predict_proba(X_text_dev) [:, stress_idxs].sum(1)
tp_te  = best_text_model.predict_proba(X_text_te)  [:, stress_idxs].sum(1)

pp_tr  = rf_physio.predict_proba(Xp_tr_uf)[:,1]
pp_dev = rf_physio.predict_proba(Xp_dev_uf)[:,1]
pp_te2 = rf_physio.predict_proba(Xp_te_uf)[:,1]

y_train_bin = np.isin(y_train, stress_idxs).astype(int)
y_dev_bin   = np.isin(y_dev,   stress_idxs).astype(int)
y_test_bin  = np.isin(y_test,  stress_idxs).astype(int)

# ─── 5️⃣ TRAIN META‐LEARNER ON TRAIN ONLY & TUNE ON DEV ───────────────────
X_meta_tr   = np.vstack([tp_tr,  pp_tr ]).T
X_meta_dev  = np.vstack([tp_dev, pp_dev]).T

meta = LogisticRegression(class_weight='balanced', random_state=42)
meta.fit(X_meta_tr, y_train_bin)

# threshold tune on dev
probs_dev,best_thr,best_f1 = meta.predict_proba(X_meta_dev)[:,1], 0.5, 0
for thr in np.linspace(0.1,0.9,81):
    p = (probs_dev>thr).astype(int)
    f1 = f1_score(y_dev_bin,p)
    if f1>best_f1:
        best_f1,best_thr = f1,thr
print(f"→ Meta threshold on dev: {best_thr:.2f} (F1={best_f1:.3f})")

# 6️⃣ EVALUATE ON TEST
X_meta_te = np.vstack([tp_te, pp_te2]).T
pred_te   = (meta.predict_proba(X_meta_te)[:,1] > best_thr).astype(int)

print("\n[Late-Fusion Meta‐Learner – Test]")
print("Accuracy:", accuracy_score(y_test_bin, pred_te))
print("F1-Score:", f1_score(y_test_bin, pred_te))
print(classification_report(y_test_bin, pred_te, target_names=["non-stress","stress"]))


KeyboardInterrupt: 

In [4]:
# INSTALL dependencies
!pip install --quiet lightgbm scipy scikit-learn tqdm pandas numpy

# ✅ Multimodal Emotion Detection (Text + Physio)
import os, glob, pickle, warnings
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal import butter, filtfilt, resample_poly
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
import lightgbm as lgb

warnings.filterwarnings("ignore")

# === Constants ===
FS_ORIG, FS, WIN_S, STRIDE_S = 700, 64, 4, 2
WIN, STRIDE = FS * WIN_S, FS * STRIDE_S

# --- Signal Processing Helpers ---
def convert_ecg(x):    return ((x/65536.0)-0.5)*3000.0
def convert_eda(x):    return ((x/65536.0)*3.0)/0.12
def convert_resp(x):   return ((x/65536.0)-0.5)*100.0

def downsample(sig, conv):
    y = conv(sig.astype(np.float32))
    cutoff, nyq = FS*0.5, FS_ORIG*0.5
    if cutoff < nyq and y.size > 8:
        b,a = butter(4, cutoff/nyq, btype='low')
        try: y = filtfilt(b,a,y)
        except: pass
    return resample_poly(y, FS, FS_ORIG)

# --- Build Physiological Features (WESAD) ---
def build_physio_feats(wesad_path, N2, N3):
    X_list, y_list = [], []
    for pkl_fp in glob.glob(os.path.join(wesad_path, 'S*', 'S*.pkl')):
        with open(pkl_fp, 'rb') as f:
            D = pickle.load(f, encoding='latin1')
        ecg  = downsample(D['signal']['chest']['ECG'],  convert_ecg)
        eda  = downsample(D['signal']['chest']['EDA'],  convert_eda)
        resp = downsample(D['signal']['chest']['Resp'], convert_resp)
        lblf = resample_poly(D['label'].astype(np.float32), FS, FS_ORIG)
        lbl  = np.rint(lblf).astype(int)
        L = min(len(ecg), len(eda), len(resp), len(lbl))
        for i in range(0, L-WIN+1, STRIDE):
            lab,_ = Counter(lbl[i:i+WIN]).most_common(1)[0]
            if lab not in (1,2,3,4): continue
            X_list.append(np.concatenate([ecg[i:i+WIN], eda[i:i+WIN], resp[i:i+WIN]]))
            y_list.append(lab-1)
    X = np.squeeze(np.stack(X_list).astype(np.float32))
    y = np.array(y_list)
    # balance classes
    max_count = max(Counter(y).values())
    X_bal, y_bal = [], []
    for cls in np.unique(y):
        Xu,yu = resample(X[y==cls], y[y==cls], replace=True, n_samples=max_count, random_state=42)
        X_bal.append(Xu); y_bal.append(yu)
    X_bal, y_bal = np.vstack(X_bal), np.concatenate(y_bal)
    # split
    X_tr, X_rest, y_tr, y_rest = train_test_split(
        X_bal, y_bal, test_size=(N2+N3)/len(y_bal), stratify=y_bal, random_state=42
    )
    X_dev, X_te = np.split(X_rest, [N2])
    # train LightGBM ensemble
    dt = lgb.LGBMClassifier(n_estimators=1, max_depth=12, random_state=42).fit(X_tr, y_tr)
    rf = lgb.LGBMClassifier(n_estimators=300, random_state=42).fit(X_tr, y_tr)
    p_dev = (dt.predict_proba(X_dev) + rf.predict_proba(X_dev)) / 2
    p_te  = (dt.predict_proba(X_te)  + rf.predict_proba(X_te))  / 2
    return p_dev, p_te

# --- Extract MELD Text Features ---
def extract_meld(data, total):
    feats = [arr for split in data for arr in split.values()]
    feats = np.vstack(feats)
    if feats.shape[0] > total:
        return feats[:total]
    pad = total - feats.shape[0]
    return np.vstack([feats, np.zeros((pad, feats.shape[1]))])

# --- Select Best Text Model ---
def best_text_model(Xtr, Xdev, Xtst, ytr, ydev, ytst):
    rf  = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
    rf.fit(np.vstack([Xtr,Xdev]), np.hstack([ytr,ydev]))
    mlp = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=200, random_state=42)
    mlp.fit(np.vstack([Xtr,Xdev]), np.hstack([ytr,ydev]))
    stack = StackingClassifier([
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
        ('rf', rf), ('mlp', mlp), ('svm', SVC(kernel='rbf', probability=True, class_weight='balanced'))
    ], final_estimator=LogisticRegression(), cv=3).fit(np.vstack([Xtr,Xdev]), np.hstack([ytr,ydev]))
    preds = { 'rf': rf.predict(Xtst), 'mlp': mlp.predict(Xtst), 'stack': stack.predict(Xtst) }
    best = max(preds, key=lambda k: f1_score(ytst, preds[k], average='weighted'))
    return {'rf':rf,'mlp':mlp,'stack':stack}[best]

# --- Full Two-Modal Fusion Pipeline ---
def run_pipeline():
    # paths
    AT    = "/kaggle/input/audiotext"
    PHYS  = "/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"
    # load MELD splits
    tr = pd.read_csv(f"{AT}/train_sent_emo.csv")
    dv = pd.read_csv(f"{AT}/dev_sent_emo.csv")
    te = pd.read_csv(f"{AT}/test_sent_emo.csv")
    emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
    y_tr = tr['Emotion'].map(emo_map).values
    y_dev= dv['Emotion'].map(emo_map).values
    y_te = te['Emotion'].map(emo_map).values
    N1,N2,N3 = len(y_tr), len(y_dev), len(y_te)
    # text features & model
    with open(f"{AT}/text_emotion.pkl", 'rb') as f: text_data = pickle.load(f)
    X_txt = extract_meld(text_data, N1+N2+N3)
    Xt_tr, Xt_dev, Xt_te = np.split(StandardScaler().fit_transform(X_txt), [N1, N1+N2])
    text_model = best_text_model(Xt_tr, Xt_dev, Xt_te, y_tr, y_dev, y_te)
    # physio features & model
    pp_dev, pp_te = build_physio_feats(PHYS, N2, N3)
    # binary stress indices
    stress = [2,3,5,6]
    # compute modality scores
    t_dev = text_model.predict_proba(Xt_dev)[:, stress].sum(axis=1)
    p_dev = pp_dev[:,1]
    t_te  = text_model.predict_proba(Xt_te)[:, stress].sum(axis=1)
    p_te  = pp_te[:,1]
    y_dev_bin = np.isin(y_dev, stress).astype(int)
    y_te_bin  = np.isin(y_te,  stress).astype(int)
    # grid search weights
    best = (0,0,-1)
    for wt in np.linspace(0,1,21):
        wp = 1 - wt
        fused = wt*t_dev + wp*p_dev
        f = f1_score(y_dev_bin, (fused>0.5).astype(int))
        if f > best[2]: best = (wt, wp, f)
    wt, wp, _ = best
    fused_te = wt*t_te + wp*p_te
    preds = (fused_te > 0.5).astype(int)
    # results
    print("\n=== Two-Modal Fusion Results ===")
    print(f"Weights → Text: {wt:.2f}, Physio: {wp:.2f}")
    print("Accuracy:", accuracy_score(y_te_bin, preds))
    print("F1-Score:", f1_score(y_te_bin, preds))
    print(classification_report(y_te_bin, preds, target_names=['non-stress','stress']))

if __name__=='__main__':
    run_pipeline()


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.181174 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 31497, number of used features: 768
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from score -1.386199
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.157005 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 31497, number of used features: 768
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from score -1.386199
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from 

In [4]:
# INSTALL dependencies
!pip install --quiet lightgbm scipy scikit-learn tqdm pandas numpy

# ✅ Multimodal Emotion Detection Script with Text, Audio (MELD), and Physio (WESAD + LightGBM)
import os, glob, pickle, warnings
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal import butter, filtfilt, resample_poly
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
import lightgbm as lgb

warnings.filterwarnings("ignore")

# === Constants ===
FS_ORIG, FS, WIN_S, STRIDE_S = 700, 64, 4, 2
WIN, STRIDE = FS * WIN_S, FS * STRIDE_S

# === Signal Processing Helpers ===
def convert_ecg(x):    return ((x/65536.0)-0.5)*3000.0

def convert_eda(x):    return ((x/65536.0)*3.0)/0.12

def convert_resp(x):   return ((x/65536.0)-0.5)*100.0

def downsample(sig, conv):
    """
    Low-pass filter and downsample signal from FS_ORIG to FS, applying conversion.
    """
    y = conv(sig.astype(np.float32))
    cutoff, nyq = FS * 0.5, FS_ORIG * 0.5
    if cutoff < nyq and y.size > 8:
        b,a = butter(4, cutoff/nyq, btype='low')
        try:
            y = filtfilt(b, a, y)
        except:
            pass
    return resample_poly(y, FS, FS_ORIG)

# === Build Physiological Features with LightGBM ===
def build_lightgbm_physio(wesad_path, N2, N3):
    """
    Loads WESAD pickles, extracts ECG, EDA, Resp windows, trains LightGBM shallow tree + forest,
    returns soft-voted probabilities on dev and test splits.
    """
    X_list, y_list = [], []
    for pkl_fp in glob.glob(os.path.join(wesad_path, "S*", "S*.pkl")):
        with open(pkl_fp, 'rb') as f:
            D = pickle.load(f, encoding='latin1')
        # Downsample chest signals
        ecg  = downsample(D['signal']['chest']['ECG'],  convert_ecg)
        eda  = downsample(D['signal']['chest']['EDA'],  convert_eda)
        resp = downsample(D['signal']['chest']['Resp'], convert_resp)
        # Resample labels
        lblf = resample_poly(D['label'].astype(np.float32), FS, FS_ORIG)
        lbl  = np.rint(lblf).astype(np.int32)
        # Align lengths and sliding windows
        L = min(len(ecg), len(eda), len(resp), len(lbl))
        for i in range(0, L-WIN+1, STRIDE):
            block = lbl[i:i+WIN]
            lab, _ = Counter(block).most_common(1)[0]
            # Only baseline=1, stress=2, amusement=3, meditation=4
            if lab not in (1,2,3,4): continue
            vec = np.concatenate([ecg[i:i+WIN], eda[i:i+WIN], resp[i:i+WIN]])
            X_list.append(vec)
            y_list.append(lab-1)
    # Stack and squeeze extra dims
    X = np.squeeze(np.stack(X_list).astype(np.float32))
    y = np.array(y_list, dtype=np.int32)
    # Balance classes via oversampling
    max_count = max(Counter(y).values())
    X_bal, y_bal = [], []
    for cls in np.unique(y):
        Xc, yc = X[y==cls], y[y==cls]
        Xu, yu = resample(Xc, yc, replace=True, n_samples=max_count, random_state=42)
        X_bal.append(Xu); y_bal.append(yu)
    X_bal = np.vstack(X_bal); y_bal = np.concatenate(y_bal)
    # Train/dev/test split on balanced data
    X_tr, X_rest, y_tr, y_rest = train_test_split(
        X_bal, y_bal, test_size=(N2+N3)/len(y_bal), stratify=y_bal, random_state=42
    )
    X_dev, X_te = np.split(X_rest, [N2])
    # LightGBM models
    dt = lgb.LGBMClassifier(n_estimators=1, max_depth=12, random_state=42).fit(X_tr, y_tr)
    rf = lgb.LGBMClassifier(n_estimators=300, random_state=42).fit(X_tr, y_tr)
    # Soft-vote
    p_dev = (dt.predict_proba(X_dev) + rf.predict_proba(X_dev)) / 2.0
    p_te  = (dt.predict_proba(X_te)  + rf.predict_proba(X_te))  / 2.0
    return p_dev, p_te

# === Extract MELD Features ===
def extract_meld(data, total):
    """
    Flatten list-of-dicts into array, pad/truncate to 'total' rows.
    """
    feats = [arr for split in data for arr in split.values()]
    feats = np.vstack(feats)
    if feats.shape[0] > total:
        return feats[:total]
    else:
        pad = total - feats.shape[0]
        return np.vstack([feats, np.zeros((pad, feats.shape[1]))])

# === Model Selection Helper ===
def best_model(Xtr, Xdev, Xtst, ytr, ydev, ytst):
    """
    Train RF, MLP, Stacking on (train+dev), pick best by weighted-F1 on test.
    """
    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=300, class_weight='balanced', random_state=42
    ).fit(np.vstack([Xtr,Xdev]), np.hstack([ytr,ydev]))
    # MLP
    mlp = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=200, random_state=42)
    mlp.fit(np.vstack([Xtr,Xdev]), np.hstack([ytr,ydev]))
    # Stacked
    stack = StackingClassifier(
        estimators=[
            ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
            ('rf', rf), ('mlp', mlp),
            ('svm', SVC(kernel='rbf', probability=True, class_weight='balanced'))
        ], final_estimator=LogisticRegression(), cv=3
    ).fit(np.vstack([Xtr,Xdev]), np.hstack([ytr,ydev]))
    # Evaluate
    candidates = {
        'rf':    rf.predict(Xtst),
        'mlp':   mlp.predict(Xtst),
        'stack': stack.predict(Xtst)
    }
    f1s = {k: f1_score(ytst, v, average='weighted') for k,v in candidates.items()}
    return {'rf': rf, 'mlp': mlp, 'stack': stack}[max(f1s, key=f1s.get)]

# === Full Pipeline Execution ===
def run_pipeline():
    # Paths
    AT    = "/kaggle/input/audiotext"
    WESAD = "/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"
    # Load MELD CSVs
    train = pd.read_csv(f"{AT}/train_sent_emo.csv")
    dev   = pd.read_csv(f"{AT}/dev_sent_emo.csv")
    test  = pd.read_csv(f"{AT}/test_sent_emo.csv")
    emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
    y_train = train['Emotion'].map(emo_map).values
    y_dev   = dev['Emotion'].map(emo_map).values
    y_test  = test['Emotion'].map(emo_map).values
    N1, N2, N3 = len(train), len(dev), len(test)
    # Load pickles
    with open(f"{AT}/text_emotion.pkl", 'rb')  as f: text_data  = pickle.load(f)
    with open(f"{AT}/audio_emotion.pkl", 'rb') as f: audio_data = pickle.load(f)
    # Extract & split
    X_text  = extract_meld(text_data,  N1+N2+N3)
    X_audio = extract_meld(audio_data, N1+N2+N3)
    Xt_tr, Xt_dev, Xt_te = np.split(StandardScaler().fit_transform(X_text),  [N1, N1+N2])
    Xa_tr, Xa_dev, Xa_te = np.split(StandardScaler().fit_transform(X_audio), [N1, N1+N2])
    # Train unimodal
    text_model  = best_model(Xt_tr, Xt_dev, Xt_te, y_train, y_dev, y_test)
    audio_model = best_model(Xa_tr, Xa_dev, Xa_te, y_train, y_dev, y_test)
    # Train physio & get probs
    pp_dev, pp_te = build_lightgbm_physio(WESAD, N2, N3)
    # Fusion - stress detection
    stress_idxs = [2,3,5,6]
    tp_dev = text_model.predict_proba(Xt_dev)[:, stress_idxs].sum(axis=1)
    ap_dev = audio_model.predict_proba(Xa_dev)[:, stress_idxs].sum(axis=1)
    tp_te  = text_model.predict_proba(Xt_te)[:, stress_idxs].sum(axis=1)
    ap_te  = audio_model.predict_proba(Xa_te)[:, stress_idxs].sum(axis=1)
    pp_dev = pp_dev[:,1];    pp_te = pp_te[:,1]
    y_dev_bin = np.isin(y_dev, stress_idxs).astype(int)
    y_te_bin  = np.isin(y_test, stress_idxs).astype(int)
    # Grid-search fusion weights
    best = (0,0,0,-1)
    for wt in np.linspace(0,1,21):
        for wa in np.linspace(0,1,21):
            wp = 1 - wt - wa
            if wp<0 or wp>1: continue
            fused = wt*tp_dev + wa*ap_dev + wp*pp_dev
            f1 = f1_score(y_dev_bin, (fused>0.5).astype(int))
            if f1 > best[3]: best = (wt, wa, wp, f1)
    wt, wa, wp, _ = best
    fused_te = wt*tp_te + wa*ap_te + wp*pp_te
    pred_te  = (fused_te>0.5).astype(int)
    # Results
    print("\n[Fused Emotion Detection: Stress vs Non-Stress]")
    print(f"Weights → Text: {wt:.2f}, Audio: {wa:.2f}, Physio: {wp:.2f}")
    print("Accuracy:", accuracy_score(y_te_bin, pred_te))
    print("F1-Score:", f1_score(y_te_bin, pred_te))
    print(classification_report(y_te_bin, pred_te, target_names=["non-stress","stress"]))

run_pipeline()


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.190396 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 31497, number of used features: 768
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from score -1.386199
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.187836 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 31497, number of used features: 768
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from score -1.386199
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from 

In [10]:
# INSTALL dependencies
!pip install --quiet lightgbm scipy scikit-learn tqdm pandas numpy

# ✅ Multimodal Emotion Detection (Text + MELD Audio + Physio)
import os, glob, pickle, warnings
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal import butter, filtfilt, resample_poly
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
import lightgbm as lgb

warnings.filterwarnings("ignore")

# === Constants ===
FS_ORIG, FS, WIN_S, STRIDE_S = 700, 64, 4, 2
WIN, STRIDE = FS * WIN_S, FS * STRIDE_S

# --- WESAD Signal Processing Helpers ---
def convert_ecg(x):    return ((x/65536.0)-0.5)*3000.0
def convert_eda(x):    return ((x/65536.0)*3.0)/0.12
def convert_resp(x):   return ((x/65536.0)-0.5)*100.0

def downsample(sig, conv):
    y = conv(sig.astype(np.float32))
    cutoff, nyq = FS*0.5, FS_ORIG*0.5
    if cutoff < nyq and y.size > 8:
        b,a = butter(4, cutoff/nyq, btype='low')
        try: y = filtfilt(b,a,y)
        except: pass
    return resample_poly(y, FS, FS_ORIG)

# --- Build Physiological Features (WESAD) ---
def build_physio_feats(wesad_path, N2, N3):
    X_list, y_list = [], []
    for pkl_fp in glob.glob(os.path.join(wesad_path, 'S*', 'S*.pkl')):
        with open(pkl_fp, 'rb') as f:
            D = pickle.load(f, encoding='latin1')
        ecg  = downsample(D['signal']['chest']['ECG'],  convert_ecg)
        eda  = downsample(D['signal']['chest']['EDA'],  convert_eda)
        resp = downsample(D['signal']['chest']['Resp'], convert_resp)
        lblf = resample_poly(D['label'].astype(np.float32), FS, FS_ORIG)
        lbl  = np.rint(lblf).astype(int)
        L = min(len(ecg), len(eda), len(resp), len(lbl))
        for i in range(0, L-WIN+1, STRIDE):
            lab,_ = Counter(lbl[i:i+WIN]).most_common(1)[0]
            if lab not in (1,2,3,4): continue
            X_list.append(np.concatenate([ecg[i:i+WIN], eda[i:i+WIN], resp[i:i+WIN]]))
            y_list.append(lab-1)
    X = np.squeeze(np.stack(X_list).astype(np.float32))
    y = np.array(y_list)
    # balance classes
    max_count = max(Counter(y).values())
    X_bal, y_bal = [], []
    for cls in np.unique(y):
        Xu,yu = resample(X[y==cls], y[y==cls], replace=True, n_samples=max_count, random_state=42)
        X_bal.append(Xu); y_bal.append(yu)
    X_bal, y_bal = np.vstack(X_bal), np.concatenate(y_bal)
    # split
    X_tr, X_rest, y_tr, y_rest = train_test_split(
        X_bal, y_bal, test_size=(N2+N3)/len(y_bal), stratify=y_bal, random_state=42
    )
    X_dev, X_te = np.split(X_rest, [N2])
    # train LightGBM ensemble
    dt = lgb.LGBMClassifier(n_estimators=1, max_depth=12, random_state=42).fit(X_tr, y_tr)
    rf = lgb.LGBMClassifier(n_estimators=300, random_state=42).fit(X_tr, y_tr)
    p_dev = (dt.predict_proba(X_dev) + rf.predict_proba(X_dev)) / 2
    p_te  = (dt.predict_proba(X_te)  + rf.predict_proba(X_te))  / 2
    return p_dev, p_te

# --- Extract MELD Unimodal Features ---
def extract_meld(data, total):
    feats = [arr for split in data for arr in split.values()]
    feats = np.vstack(feats)
    if feats.shape[0] > total:
        return feats[:total]
    pad = total - feats.shape[0]
    return np.vstack([feats, np.zeros((pad, feats.shape[1]))])

# --- Select Best Unimodal Model ---
def select_best(Xtr, Xdev, Xtst, ytr, ydev, ytst):
    rf  = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
    rf.fit(np.vstack([Xtr,Xdev]), np.hstack([ytr,ydev]))
    mlp = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=200, random_state=42)
    mlp.fit(np.vstack([Xtr,Xdev]), np.hstack([ytr,ydev]))
    stack = StackingClassifier([
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
        ('rf', rf), ('mlp', mlp), ('svm', SVC(kernel='rbf', probability=True, class_weight='balanced'))
    ], final_estimator=LogisticRegression(), cv=3).fit(np.vstack([Xtr,Xdev]), np.hstack([ytr,ydev]))
    preds = {'rf':rf.predict(Xtst), 'mlp':mlp.predict(Xtst), 'stack':stack.predict(Xtst)}
    best_key = max(preds, key=lambda k: f1_score(ytst, preds[k], average='weighted'))
    return {'rf':rf, 'mlp':mlp, 'stack':stack}[best_key]

# --- Full Three-Modal Fusion Pipeline ---
def run_pipeline():
    AT    = "/kaggle/input/audiotext"
    PHYS  = "/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"
    # 1) Load MELD splits & labels
    tr = pd.read_csv(f"{AT}/train_sent_emo.csv")
    dv = pd.read_csv(f"{AT}/dev_sent_emo.csv")
    te = pd.read_csv(f"{AT}/test_sent_emo.csv")
    emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
    y_tr = tr['Emotion'].map(emo_map).values
    y_dev= dv['Emotion'].map(emo_map).values
    y_te = te['Emotion'].map(emo_map).values
    N1,N2,N3 = len(y_tr), len(y_dev), len(y_te)

    # 2) MELD text branch
    with open(f"{AT}/text_emotion.pkl", 'rb') as f: text_data = pickle.load(f)
    X_txt = extract_meld(text_data, N1+N2+N3)
    Xt_tr, Xt_dev, Xt_te = np.split(StandardScaler().fit_transform(X_txt), [N1, N1+N2])
    mdl_txt = select_best(Xt_tr, Xt_dev, Xt_te, y_tr, y_dev, y_te)

    # 3) MELD audio branch
    with open(f"{AT}/audio_emotion.pkl", 'rb') as f: audio_data = pickle.load(f)
    X_aud = extract_meld(audio_data, N1+N2+N3)
    Xa_tr, Xa_dev, Xa_te = np.split(StandardScaler().fit_transform(X_aud), [N1, N1+N2])
    mdl_aud = select_best(Xa_tr, Xa_dev, Xa_te, y_tr, y_dev, y_te)

    # 4) WESAD physio branch
    pp_dev, pp_te = build_physio_feats(PHYS, N2, N3)

        # 5) Late fusion (stress vs non-stress)
    stress_idxs = [2,3,5,6]
    y_dev_bin = np.isin(y_dev, stress_idxs).astype(int)
    y_te_bin  = np.isin(y_te,  stress_idxs).astype(int)

    t_dev = mdl_txt.predict_proba(Xt_dev)[:, stress_idxs].sum(axis=1)
    a_dev = mdl_aud.predict_proba(Xa_dev)[:, stress_idxs].sum(axis=1)
    p_dev = pp_dev[:,1]

    t_te  = mdl_txt.predict_proba(Xt_te)[:, stress_idxs].sum(axis=1)
    a_te  = mdl_aud.predict_proba(Xa_te)[:, stress_idxs].sum(axis=1)
    p_te  = pp_te[:,1]

    best = (0.0, 0.0, 0.0, -1.0)
    for wt in np.linspace(0,1,11):
        for wa in np.linspace(0,1,11):
            if wa <= 0.0:
                continue
            wp = 1.0 - wt - wa
            if wp < 0.0 or wp > 1.0:
                continue
            fused = wt * t_dev + wa * a_dev + wp * p_dev
            f1v = f1_score(y_dev_bin, (fused > 0.5).astype(int))
            if f1v > best[3]:
                best = (wt, wa, wp, f1v)
    wt, wa, wp, _ = best

    fused_te = wt * t_te + wa * a_te + wp * p_te
    preds    = (fused_te > 0.5).astype(int)

    print("=== Three-Modal Fusion Results ===")
    print(f"Weights → Text: {wt:.2f}, Audio: {wa:.2f}, Physio: {wp:.2f}")
    print("Accuracy:", accuracy_score(y_te_bin, preds))
    print("F1-Score:", f1_score(y_te_bin, preds))
    print(classification_report(
        y_te_bin, preds,
        target_names=['non-stress','stress']
    ))


if __name__=='__main__':
    run_pipeline()


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.187915 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 31497, number of used features: 768
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from score -1.386199
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.184692 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 31497, number of used features: 768
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from score -1.386199
[LightGBM] [Info] Start training from score -1.386326
[LightGBM] [Info] Start training from 

In [7]:
# INSTALL dependencies
!pip install --quiet lightgbm scipy scikit-learn tqdm pandas numpy

# ✅ Three-Modal Fusion: MELD Text + MELD Audio + WESAD Physio (Fixed Physio Builder)
import os, glob, pickle, warnings
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal import butter, filtfilt, resample_poly, find_peaks
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
import lightgbm as lgb

warnings.filterwarnings("ignore")

# === Constants ===
FS_ORIG, FS_E, FS_R = 700, 64, 4  # ECG @64Hz, EDA/Resp @4Hz
WIN_SEC, STRIDE_SEC = 4, 2
WIN_E, STRIDE_E = FS_E*WIN_SEC, FS_E*STRIDE_SEC
WIN_R, STRIDE_R = FS_R*WIN_SEC, FS_R*STRIDE_SEC

# --- Physiologic Feature Extraction (from original WESAD script) ---

def convert_ecg(x):    return ((x/65536.0)-0.5)*3000.0

def convert_eda(x):    return ((x/65536.0)*3.0)/0.12

def convert_resp(x):   return ((x/65536.0)-0.5)*100.0


def lowpass(sig, cutoff, fs):
    nyq = fs*0.5
    if cutoff>=nyq or sig.size<10:
        return sig
    b,a = butter(4, cutoff/nyq, btype='low')
    try:
        return filtfilt(b,a,sig)
    except:
        return sig


def downsample(sig, from_fs, to_fs, conv=None):
    if conv: sig = conv(sig)
    if from_fs==to_fs: return sig
    sig = lowpass(sig, to_fs*0.5, from_fs)
    return resample_poly(sig, to_fs, from_fs)


def basic_stats(x):
    x = x.flatten()
    return [
        x.mean(), x.std(), x.min(), x.max(),
        np.percentile(x,25), np.percentile(x,50), np.percentile(x,75),
        x.ptp(), np.mean(np.abs(np.diff(x)))
    ]

def ecg_feats(x, fs):
    x = x.flatten()
    peaks,_ = find_peaks(x, distance=fs*0.5)
    rr = np.diff(peaks)/fs if len(peaks)>1 else np.array([])
    hr  = 60/rr.mean() if rr.size>0 else 0
    hrv = rr.std()    if rr.size>0 else 0
    return [hr, hrv] + basic_stats(x)


def build_physio(root="/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"):
    # Collect features & labels
    Xp, yp = [], []
    for pkl_fp in glob.glob(os.path.join(root, "S*", "S*.pkl")):
        with open(pkl_fp,'rb') as f:
            d = pickle.load(f, encoding='latin1')
        ecg700 = d['signal']['chest']['ECG']
        eda700 = d['signal']['chest']['EDA']
        resp700= d['signal']['chest']['Resp']
        lbl700 = d['label']
        # downsample
        ecg_ds  = downsample(ecg700, FS_ORIG, FS_E,    convert_ecg)
        eda_ds  = downsample(eda700,FS_ORIG, FS_R,    convert_eda)
        resp_ds = downsample(resp700,FS_ORIG, FS_R,   convert_resp)
        win_e, str_e = WIN_E, STRIDE_E
        win_r, str_r = WIN_R, STRIDE_R
        for i in range(0, len(lbl700)-FS_ORIG*WIN_SEC+1, FS_ORIG*STRIDE_SEC):
            widx = slice(i, i+FS_ORIG*WIN_SEC)
            lab,_ = Counter(lbl700[widx]).most_common(1)[0]
            if lab not in (1,2,3,4): continue
            # map to ds indices
            e0,e1 = int(i/FS_ORIG*FS_E), int((i+FS_ORIG*WIN_SEC)/FS_ORIG*FS_E)
            r0,r1 = int(i/FS_ORIG*FS_R), int((i+FS_ORIG*WIN_SEC)/FS_ORIG*FS_R)
            feats = ecg_feats(ecg_ds[e0:e1], FS_E) + basic_stats(eda_ds[r0:r1]) + basic_stats(resp_ds[r0:r1])
            Xp.append(feats); yp.append(lab-1)
    return np.array(Xp), np.array(yp)


def build_physio_rf(root, test_frac=0.3):
    Xp, yp = build_physio(root)
    X_tr, X_te, y_tr, y_te = train_test_split(
        Xp, yp, test_size=test_frac,
        stratify=yp, random_state=42
    )
    rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
    rf.fit(X_tr, y_tr)
    return rf.predict_proba(X_tr), rf.predict_proba(X_te)

# --- MELD Text / Audio Helper ---
def extract_meld(data, total):
    arr = [v for split in data for v in split.values()]
    feats = np.vstack(arr)
    if feats.shape[0] > total:
        return feats[:total]
    pad = total - feats.shape[0]
    return np.vstack([feats, np.zeros((pad, feats.shape[1]))])

def select_best(X_tr, X_dev, X_te, y_tr, y_dev, y_te):
    rf  = RandomForestClassifier(300, class_weight='balanced', random_state=42)
    rf .fit(np.vstack([X_tr,X_dev]), np.hstack([y_tr,y_dev]))
    mlp = MLPClassifier((256,128),max_iter=200,random_state=42)
    mlp.fit(np.vstack([X_tr,X_dev]), np.hstack([y_tr,y_dev]))
    stack=StackingClassifier([
        ('lr',LogisticRegression(max_iter=1000,class_weight='balanced')),
        ('rf',rf),('mlp',mlp),('svm',SVC(kernel='rbf',probability=True,class_weight='balanced'))
    ], final_estimator=LogisticRegression(),cv=3)
    stack.fit(np.vstack([X_tr,X_dev]), np.hstack([y_tr,y_dev]))
    preds = {k:m.predict(X_te) for k,m in [('rf',rf),('mlp',mlp),('stack',stack)]}
    best = max(preds, key=lambda k: f1_score(y_te, preds[k], average='weighted'))
    return {'rf':rf,'mlp':mlp,'stack':stack}[best]

# --- Full Three-Modal Fusion Pipeline ---
def run_pipeline():
    AT = "/kaggle/input/audiotext"
    WESAD = "/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"
    # 1) MELD splits
    tr = pd.read_csv(f"{AT}/train_sent_emo.csv")
    dv = pd.read_csv(f"{AT}/dev_sent_emo.csv")
    te = pd.read_csv(f"{AT}/test_sent_emo.csv")
    emo_map={'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
    ytr = tr['Emotion'].map(emo_map).values
    ydv = dv['Emotion'].map(emo_map).values
    yte = te['Emotion'].map(emo_map).values
    N1,N2,N3=len(ytr),len(ydv),len(yte)
    # 2) Text
    with open(f"{AT}/text_emotion.pkl",'rb') as f: td=pickle.load(f)
    Xtxt=extract_meld(td,N1+N2+N3)
    Xt_tr,Xt_dv,Xt_te=np.split(StandardScaler().fit_transform(Xtxt),[N1,N1+N2])
    mdl_txt=select_best(Xt_tr,Xt_dv,Xt_te,ytr,ydv,yte)
    # 3) Audio
    with open(f"{AT}/audio_emotion.pkl",'rb') as f: ad=pickle.load(f)
    Xad=extract_meld(ad,N1+N2+N3)
    Xa_tr,Xa_dv,Xa_te=np.split(StandardScaler().fit_transform(Xad),[N1,N1+N2])
    mdl_aud=select_best(Xa_tr,Xa_dv,Xa_te,ytr,ydv,yte)
    # 4) Physio
    pp_tr,pp_te = build_physio_rf(WESAD)
    # align physio to dev/test lengths
    p_dev = pp_tr[:N2,1]
    p_te  = pp_te[:N3,1]
    # 5) Fusion
    idx=[2,3,5,6]
    t_dev=mdl_txt.predict_proba(Xt_dv)[:,idx].sum(1)
    a_dev=mdl_aud.predict_proba(Xa_dv)[:,idx].sum(1)
    ydv_bin=np.isin(ydv,idx).astype(int)
    # grid-search
    best=(0,0,0,-1)
    for wt in np.linspace(0,1,21):
        for wa in np.linspace(0,1,21):
            wp=1-wt-wa
            if wp<0 or wp>1: continue
            f= f1_score(ydv_bin,((wt*t_dev+wa*a_dev+wp*p_dev)>0.5).astype(int))
            if f>best[3]: best=(wt,wa,wp,f)
    wt,wa,wp,_=best
    t_te=mdl_txt.predict_proba(Xt_te)[:,idx].sum(1)
    a_te=mdl_aud.predict_proba(Xa_te)[:,idx].sum(1)
    preds=((wt*t_te+wa*a_te+wp*p_te)>0.5).astype(int)
    yte_bin=np.isin(yte,idx).astype(int)
    print("\n=== Fusion Results ===")
    print(f"W → Txt:{wt:.2f},Aud:{wa:.2f},Phy:{wp:.2f}")
    print("Acc:",accuracy_score(yte_bin,preds))
    print("F1:",f1_score(yte_bin,preds))
    print(classification_report(yte_bin,preds,target_names=['non-str','str']))

if __name__=='__main__': run_pipeline()



=== Fusion Results ===
W → Txt:0.80,Aud:0.05,Phy:0.15
Acc: 0.7413793103448276
F1: 0.03708987161198288
              precision    recall  f1-score   support

     non-str       0.74      0.99      0.85      1939
         str       0.43      0.02      0.04       671

    accuracy                           0.74      2610
   macro avg       0.59      0.51      0.44      2610
weighted avg       0.66      0.74      0.64      2610



In [5]:
# INSTALL dependencies
!pip install --quiet lightgbm scipy scikit-learn tqdm pandas numpy

"""
✅ Multimodal Emotion Fusion Script
- Unimodal multi-class accuracies for Text (MELD), Audio (MELD), Physio (WESAD)
- Binary stress-vs-non-stress threshold tuning per modality
- Late-fusion grid search with minimum weight=0.15
"""
import os, glob, pickle, warnings
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal import butter, filtfilt, resample_poly, find_peaks
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
import lightgbm as lgb

warnings.filterwarnings("ignore")

# === Constants ===
FS_ORIG = 700
FS_E, FS_R = 64, 4
WIN_SEC, STRIDE_SEC = 4, 2
WIN_E, STR_E = FS_E * WIN_SEC, FS_E * STRIDE_SEC
WIN_R, STR_R = FS_R * WIN_SEC, FS_R * STRIDE_SEC
STRESS_IDXS = [2,3,5,6]

# --- Physio Feature Extraction ---
def convert_ecg(x):    return ((x/65536.0)-0.5)*3000.0
def convert_eda(x):    return ((x/65536.0)*3.0)/0.12
def convert_resp(x):   return ((x/65536.0)-0.5)*100.0

def lowpass(sig, cutoff, fs):
    nyq = fs*0.5
    if cutoff>=nyq or sig.size<10: return sig
    b,a = butter(4, cutoff/nyq, btype='low')
    try: return filtfilt(b,a,sig)
    except: return sig

def downsample(sig, from_fs, to_fs, conv=None):
    if conv: sig = conv(sig)
    if from_fs==to_fs: return sig
    sig = lowpass(sig, to_fs*0.5, from_fs)
    return resample_poly(sig, to_fs, from_fs)

def basic_stats(x):
    x=x.flatten()
    return [x.mean(), x.std(), x.min(), x.max(),
            np.percentile(x,25), np.percentile(x,50), np.percentile(x,75),
            x.ptp(), np.mean(np.abs(np.diff(x)))]

def ecg_feats(x,fs):
    x=x.flatten()
    peaks,_=find_peaks(x,distance=fs*0.5)
    rr=np.diff(peaks)/fs if len(peaks)>1 else np.array([])
    hr=60/rr.mean() if rr.size>0 else 0
    hrv=rr.std() if rr.size>0 else 0
    return [hr,hrv]+basic_stats(x)

# Build physio dataset
def build_physio(root):
    Xp, yp = [], []
    for f in glob.glob(os.path.join(root,'S*','S*.pkl')):
        d=pickle.load(open(f,'rb'),encoding='latin1')
        ecg=downsample(d['signal']['chest']['ECG'],FS_ORIG,FS_E,convert_ecg)
        eda=downsample(d['signal']['chest']['EDA'],FS_ORIG,FS_R,convert_eda)
        resp=downsample(d['signal']['chest']['Resp'],FS_ORIG,FS_R,convert_resp)
        lbl=d['label']
        L=len(lbl)
        for i in range(0, L-FS_ORIG*WIN_SEC+1, FS_ORIG*STRIDE_SEC):
            block=lbl[i:i+FS_ORIG*WIN_SEC]
            lab,_=Counter(block).most_common(1)[0]
            if lab not in (1,2,3,4): continue
            e0,e1=int(i/FS_ORIG*FS_E),int((i+FS_ORIG*WIN_SEC)/FS_ORIG*FS_E)
            r0,r1=int(i/FS_ORIG*FS_R),int((i+FS_ORIG*WIN_SEC)/FS_ORIG*FS_R)
            feat=ecg_feats(ecg[e0:e1],FS_E)+basic_stats(eda[r0:r1])+basic_stats(resp[r0:r1])
            Xp.append(feat); yp.append(lab-1)
    return np.array(Xp), np.array(yp)

# Train physio multi-class RF
def train_physio(root):
    Xp,yp=build_physio(root)
    Xp_tr,Xp_te,yp_tr,yp_te=train_test_split(Xp,yp,test_size=0.3,stratify=yp,random_state=42)
    rf=RandomForestClassifier(n_estimators=200,class_weight='balanced',random_state=42)
    rf.fit(Xp_tr,yp_tr)
    return Xp_tr,yp_tr,Xp_te,yp_te,rf

# MELD feature helpers
def extract_meld(data,total):
    arr=np.vstack([v for d in data for v in d.values()])
    if arr.shape[0]>=total: return arr[:total]
    return np.vstack([arr,np.zeros((total-arr.shape[0],arr.shape[1]))])

# Unimodal model trainer
def train_unimodal(X_tr,X_dev,y_tr,y_dev):
    RF=RandomForestClassifier(300,'balanced',random_state=42); RF.fit(np.vstack([X_tr,X_dev]),np.hstack([y_tr,y_dev]))
    MLP=MLPClassifier((256,128),max_iter=200,random_state=42);MLP.fit(np.vstack([X_tr,X_dev]),np.hstack([y_tr,y_dev]))
    ST=StackingClassifier([('lr',LogisticRegression(max_iter=1000,class_weight='balanced')),('rf',RF),('mlp',MLP),('svm',SVC(kernel='rbf',probability=True,class_weight='balanced'))],final_estimator=LogisticRegression(),cv=3)
    ST.fit(np.vstack([X_tr,X_dev]),np.hstack([y_tr,y_dev]))
    return RF,MLP,ST

if __name__=='__main__':
    # Paths
    AT='/kaggle/input/audiotext'
    WESAD='/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD'

    # Load MELD splits
    train_df=pd.read_csv(f'{AT}/train_sent_emo.csv')
    dev_df  =pd.read_csv(f'{AT}/dev_sent_emo.csv')
    test_df =pd.read_csv(f'{AT}/test_sent_emo.csv')
    emo_map={'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
    y_tr=train_df['Emotion'].map(emo_map).values
    y_dev=dev_df  ['Emotion'].map(emo_map).values
    y_te=test_df ['Emotion'].map(emo_map).values
    N1,N2,N3=len(y_tr),len(y_dev),len(y_te)

    # Text unimodal
    text_data=pickle.load(open(f'{AT}/text_emotion.pkl','rb'))
    X_txt=extract_meld(text_data,N1+N2+N3)
    Xt_tr,Xt_dev,Xt_te=np.split(StandardScaler().fit_transform(X_txt),[N1,N1+N2])
    RF_txt,MLP_txt,ST_txt=train_unimodal(Xt_tr,Xt_dev,y_tr,y_dev)
    # evaluate multimodal
    print('\n--- Text (7-way) Accuracy & Report ---')
    print('Accuracy:',accuracy_score(y_te,ST_txt.predict(Xt_te)))
    print(classification_report(y_te,ST_txt.predict(Xt_te)))

    # Audio unimodal
    audio_data=pickle.load(open(f'{AT}/audio_emotion.pkl','rb'))
    X_aud=extract_meld(audio_data,N1+N2+N3)
    Xa_tr,Xa_dev,Xa_te=np.split(StandardScaler().fit_transform(X_aud),[N1,N1+N2])
    RF_aud,MLP_aud,ST_aud=train_unimodal(Xa_tr,Xa_dev,y_tr,y_dev)
    print('\n--- Audio (7-way) Accuracy & Report ---')
    print('Accuracy:',accuracy_score(y_te,ST_aud.predict(Xa_te)))
    print(classification_report(y_te,ST_aud.predict(Xa_te)))

    # Physio unimodal
    Xp_tr,yp_tr,Xp_te,yp_te,RF_phy=train_physio(WESAD)
    print('\n--- Physio (4-way) Accuracy & Report ---')
    print('Accuracy:',accuracy_score(yp_te,RF_phy.predict(Xp_te)))
    print(classification_report(yp_te,RF_phy.predict(Xp_te),target_names=['baseline','stress','amuse','med']))

    # Binary stress detection per modality (threshold tuning)
    y_dev_bin=np.isin(y_dev,STRESS_IDXS).astype(int)
    y_te_bin =np.isin(y_te, STRESS_IDXS).astype(int)
    # get stress probs
    t_dev=ST_txt.predict_proba(Xt_dev)[:,STRESS_IDXS].sum(1)
    a_dev=ST_aud.predict_proba(Xa_dev)[:,STRESS_IDXS].sum(1)
    p_dev=RF_phy.predict_proba(Xp_tr)[:,1]
    # tune threshold
    def tune(probs, y):
        best=(0,0)
        for th in np.linspace(0,1,101):
            acc=accuracy_score(y,(probs>th).astype(int))
            if acc>best[1]: best=(th,acc)
        return best
    th_t,acc_t=tune(t_dev,y_dev_bin)
    th_a,acc_a=tune(a_dev,y_dev_bin)
    th_p,acc_p=tune(p_dev,ytr_p)
    print(f"\nUnimodal Binary (stress vs non) Accuracies on dev: Text={acc_t:.3f}, Audio={acc_a:.3f}, Physio={acc_p:.3f}")

    # Grid-search fusion weights >=0.15
    best=(0,0,0,-1)
    for wt in np.arange(0.15,1.01,0.05):
        for wa in np.arange(0.15,1.01,0.05):
            wp=1-wt-wa
            if wp<0.15: continue
            fused=wt*t_dev+wa*a_dev+wp*p_dev
            f1v=f1_score(y_dev_bin,(fused>0.5).astype(int))
            if f1v>best[3]: best=(wt,wa,wp,f1v)
    wt,wa,wp,_=best
    print(f"\nOptimal weights(dev F1): Text={wt:.2f},Audio={wa:.2f},Physio={wp:.2f}")

    # Fusion on test
    t_te=ST_txt.predict_proba(Xt_te)[:,STRESS_IDXS].sum(1)
    a_te=ST_aud.predict_proba(Xa_te)[:,STRESS_IDXS].sum(1)
    p_te=RF_phy.predict_proba(Xp_te)[:,1]
    fused_te=wt*t_te+wa*a_te+wp*p_te
    preds=(fused_te>0.5).astype(int)
    print("\n--- Fusion Results (stress vs non) ---")
    print('Accuracy:',accuracy_score(y_te_bin,preds))
    print('F1-Score:',f1_score(y_te_bin,preds))
    print(classification_report(y_te_bin,preds,target_names=['non-stress','stress']))


ValueError: Found input variables with inconsistent numbers of samples: [1109, 15734]